# [실습] 인과추론 - (2) DiD/RDD/PSM

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# 🔍 실험 없이 인과를 추정한다 — 이중차분과 준실험

## — DiD, 그리고 RDD·PSM 맛보기

---

이전 시간 우리는 인과의 황금 표준 — **무작위 실험(A/B 테스트)**을 배웠습니다. 그런데 현실에는 실험을 *할 수 없는* 상황이 너무 많습니다. 이미 시행된 정책, 윤리적으로 무작위 배정이 불가능한 처치, 비용이 막대한 개입… 이럴 때 관찰 데이터만으로 인과에 다가가는 **준실험(Quasi-Experiment)** 설계를 배웁니다. 그 대표 주자가 **이중차분(DiD)**입니다. 이번 시간으로 통계·인과추론 모듈을 마무리합니다.

## 📋 학습 목표

이 노트북을 끝내면 여러분은 다음을 할 수 있습니다.

1. 무작위 실험이 *불가능한* 상황에서 왜 단순 비교(전후·집단)가 위험한지 설명할 수 있습니다.
2. **이중차분(DiD)**의 직관 — "차이의 차이"로 시간 추세와 집단 차이를 동시에 걷어내는 원리를 이해합니다.
3. `statsmodels`로 **DiD를 회귀로 구현**하고 상호작용 계수에서 **ATT**를 읽어낼 수 있습니다.
4. DiD의 생명인 **평행추세 가정**을 시각화·플라시보 검정으로 점검할 수 있습니다.
5. **RDD·PSM**이 각각 어떤 상황에 맞는 도구인지 판단하고, "결론보다 가정을 먼저 검증"하는 태도를 갖춥니다.

## 📚 학습 목차

| Part | 주제 | 이 파트에서 답할 질문 | ⏱ 예상 |
|------|------|----------------------| --- |
| 1 | 실험을 할 수 없을 때 | 무작위가 불가능하면 인과는 포기해야 하나? | 5분 |
| 2 | 환경 준비와 데이터 | 무엇을, 어떤 언어로 다루나? | 5분 |
| 3 | 단순 비교의 함정 | 전후 비교·집단 비교는 왜 틀리나? | 15분 |
| 4 | 이중차분(DiD) 직관 | "차이의 차이"가 어떻게 편향을 걷어내나? | 30분 |
| 5 | DiD를 회귀로 구현 | statsmodels로 ATT를 어떻게 추정하나? | 40분 |
| 6 | 평행추세 가정 | DiD를 믿어도 되는지 어떻게 점검하나? | 45분 |
| 7 | RDD·PSM 맛보기 | 다른 준실험 도구는 언제 쓰나? | 20분 |
| 8 | 가정 먼저 검증 | 인과 분석가의 태도는 무엇인가? | 5분 |
| 🧪 | 종합 실습 | 정책 효과를 DiD로 추정·검증·보고할 수 있나? | 45분 |

*⏱ 예상 시간은 참고용 안내입니다. 학습 속도는 사람마다 다릅니다 — 여러분의 속도로 가면 됩니다.*

# Part 1. 실험을 할 수 없을 때

이전 시간 우리는 무작위 배정이 인과 추정의 황금 표준임을 배웠습니다. 무작위로 처치·대조를 나누면 두 집단이 *처치 빼고 똑같아져서*, 결과의 차이를 처치 덕분이라 말할 수 있었습니다. 그런데 현실은 종종 그 황금 표준을 허락하지 않습니다.

생각해보세요. "최저임금 인상이 고용에 미친 효과"를 알고 싶다고 무작위로 어떤 도시엔 인상하고 어떤 도시엔 안 할 수는 없습니다. 정책은 이미 *전국에 동시에* 시행됐거나, 정치적으로 결정됐으니까요. "흡연이 건강에 미친 영향"을 알겠다고 사람들을 무작위로 흡연군에 배정하는 건 *윤리적으로 불가능*합니다. "새 매장 디자인의 매출 효과"를 보겠다고 수십억을 들여 매장 절반을 무작위로 개조하기도 어렵습니다. 이렇게 **무작위 실험이 불가능하거나, 비용·윤리·시간 때문에 현실성이 없을 때**가 훨씬 많습니다.

그렇다면 인과를 포기해야 할까요? 아닙니다. 통계학자들은 *이미 일어난* 관찰 데이터에서 인과에 다가가는 영리한 설계들을 발전시켰습니다. 이를 **준실험(Quasi-Experiment)**이라 부릅니다. 무작위 배정이라는 마법은 없지만, *그럴듯한 비교 대상(대조군)을 영리하게 찾아내* 무작위 실험을 흉내 내는 것입니다. 오늘 그 대표 도구인 **이중차분(Difference-in-Differences, DiD)**을 중심으로, RDD·PSM까지 살펴봅니다.

오해를 하나 미리 풀어두겠습니다. "준실험은 무작위 실험보다 *못한* 차선책 아닌가?"라고 생각할 수 있습니다. 어떤 의미에선 맞습니다 — 무작위 배정이 주는 깨끗함에는 못 미치니까요. 하지만 준실험은 단순한 차선책 이상입니다. 현실의 수많은 중요한 질문(정책·사회·역사적 사건의 효과)은 *애초에 실험이 불가능*하기에, 준실험이 *유일하게 가능한 인과 추정*인 경우가 많거든요. 노벨 경제학상이 이런 준실험 방법론(자연실험·DiD 등)에 수여됐을 만큼, 이건 변방의 기법이 아니라 현대 실증 분석의 핵심입니다. 그러니 "실험을 못 하니 포기"가 아니라 "실험을 못 해도 *영리하게* 추정한다"는 적극적 태도로 접근해 주세요.

> ❓ **이 파트에서 답할 질문:** 무작위 실험이 불가능할 때, 관찰 데이터만으로 인과 효과에 어떻게 다가갈 수 있을까요?

> 🎯 **오늘의 핵심:** 준실험의 전부는 **"믿을 만한 대조군을 어떻게 구하느냐"**입니다. DiD는 *처치 전의 추세*를 대조군 삼아, 시간이 만든 변화와 처치가 만든 변화를 분리합니다. 그리고 그 모든 방법은 **하나의 가정 위에 서 있어요** — DiD라면 "평행추세". 그래서 오늘의 진짜 교훈은 기법이 아니라 태도입니다: **결론을 말하기 전에, 그 결론이 기댄 가정을 먼저 검증하라.**

이번 시간은 모듈의 마지막입니다. 기술통계로 데이터를 읽고, 차이를 검정하고, 관계를 모델링하고, 실험으로 인과를 확인하는 긴 여정의 끝에서, 오늘은 *실험 없이도* 인과에 다가가는 법으로 마무리합니다. 끝에 모듈 전체를 한 장으로 돌아보겠습니다.

한 가지 동기부여를 더하자면 — 준실험 인과 분석은 데이터 직무에서 *고급 역량*으로 인정받습니다. A/B 테스트조차 돌릴 수 없는 상황(이미 출시된 기능, 경쟁사 정책, 거시 환경 변화)에서 "그래서 그게 효과가 있었나?"를 *방어 가능한 방법으로* 답할 수 있는 분석가는 의외로 드물거든요. 정책 효과 평가, 마케팅 인과 측정, 제품 변경의 장기 영향 추정 등 — 실무의 까다로운 인과 질문 대부분이 관찰 데이터 위에서 벌어집니다. 오늘 배우는 DiD와 준실험 사고는, 그 어려운 질문 앞에서 "데이터로 신중하게, 그러나 분명하게" 답하는 무기가 되어줄 것입니다. 그러니 조금 어렵더라도 끝까지 따라와 주세요 — 이 모듈의 가장 값진 도구가 마지막에 있습니다.

# Part 2. 환경 준비와 데이터

오늘도 **두 종류의 데이터**를 함께 씁니다. 역할이 분명히 다릅니다.

| 데이터 | 어디에 쓰나 | 왜 그 역할인가 |
| --- | --- | --- |
| **시뮬레이션**(정답 ATT를 심은 합성 데이터) | Part 3~6의 개념 학습, 평행추세 위반 실험, RDD 시연 | 인과 효과의 *참값*은 현실에서 관측할 수 없습니다. 정답을 심어야 **DiD가 그 정답을 되찾는지**, **가정이 깨지면 얼마나 틀리는지**를 눈으로 확인할 수 있습니다. |
| **실제 정책·프로그램 데이터**(라이브러리 제공) | Part 5·6·7의 실전 적용 | 교과서 예제가 아니라 논문에 쓰인 실제 데이터로 같은 절차를 밟아, 실무 감각을 얻습니다. |

### 오늘 쓰는 실제 데이터 세 가지

| 데이터 | 무엇인가 | 어디에 쓰나 |
| --- | --- | --- |
| `injury` (`wooldridge`) | 1980년 **켄터키주 산재보험 급여 상한 인상** 전후의 재해 근로자 기록 7,150건 | **실제 정책 DiD** (Part 5) |
| `Cigar` (`plm`) | 미국 46개 주 × 1963~1992년 담배 판매량 패널 | **이벤트 스터디·평행추세** (Part 6) — 1989년 캘리포니아 담배세 인상(Proposition 99) |
| `Treatment` (`Ecdat`) | **직업훈련 프로그램**(NSW) 참가자 185명 + 비교집단 2,490명의 소득 | **PSM 실습** (Part 7) — 인과추론 문헌의 고전 데이터 |

> ⚠️ 세 데이터 모두 `sm.datasets.get_rdataset()`으로 받아오므로 **첫 실행에 인터넷 연결이 필요**합니다.
>
> 💡 **RDD는 왜 시뮬레이션인가?** RDD에는 "점수가 임계값을 넘으면 처치"라는 *기계적 배정 규칙*이 담긴 데이터가 필요합니다. 파이썬 라이브러리가 기본 제공하는 데이터셋에는 그런 구조가 없어, 오늘은 규칙을 우리가 만든 시뮬레이션으로 원리를 익히고 **실제 연구 사례를 함께 소개**합니다.

> ⚠️ **주의:** 코드 셀은 반드시 위에서부터 순서대로 실행하세요. 중간부터 실행하면 이전 셀의 변수를 찾지 못해 오류가 납니다.

## 1단계 · 환경 준비와 실제 정책 데이터 로드

In [ ]:
# ─────────────────────────────────────────────
# [C1] 환경 준비 — 라이브러리 + 한글 폰트 + 시드 고정 + 실제 정책 데이터 로드
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas scipy statsmodels matplotlib seaborn scikit-learn -q

import platform
import warnings

import numpy as np
import pandas as pd
import scipy
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression      # PSM: 성향점수 추정
from sklearn.neighbors import NearestNeighbors           # PSM: 최근접 이웃 매칭

warnings.filterwarnings("ignore")

# 재현성: 같은 난수를 항상 같게 만듭니다. (진짜 효과를 심은 합성 데이터에 적용)
np.random.seed(42)                                                # 재현성: 난수를 항상 같게 고정

# 그래프 스타일을 '먼저' 적용합니다.
# ⚠️ seaborn의 set_style()은 font.family를 sans-serif로 되돌립니다.
#    그래서 한글 폰트를 이 줄보다 앞에 설정하면 그림의 한글이 □□로 깨집니다.
sns.set_style("whitegrid")

# 한글 폰트 설정 — 반드시 sns.set_style() '뒤에' 둡니다.
system = platform.system()
if system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
else:
    # 리눅스/Colab: 한글 폰트가 없으면 아래를 먼저 실행하세요.
    # !apt-get -qq install -y fonts-nanum && rm -rf ~/.cache/matplotlib
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)

# ── 실제 정책·프로그램 데이터 (라이브러리 제공) ──
injury = sm.datasets.get_rdataset("injury", "wooldridge").data      # 켄터키 정책 DiD
cigar  = sm.datasets.get_rdataset("Cigar", "plm").data              # 담배 판매 패널(이벤트 스터디)
nsw    = sm.datasets.get_rdataset("Treatment", "Ecdat").data        # 직업훈련(PSM)

print("준비 완료!")
print("numpy       :", np.__version__)
print("scipy       :", scipy.__version__)
print("statsmodels :", sm.__version__)
print()
print(f"injury : {injury.shape}  — 켄터키(ky=1) {int((injury['ky']==1).sum())}건")
print(f"cigar  : {cigar.shape}  — {cigar['state'].nunique()}개 주 × {cigar['year'].nunique()}년")
print(f"nsw    : {nsw.shape}  — 훈련 참가 {int(nsw['treat'].sum())}명 / 비교집단 {int((~nsw['treat'].astype(bool)).sum())}명")

## 📖 핵심 용어 사전

| 기호/용어 | 읽는 법 | 의미 |
|-----------|---------|------|
| 처치군 / 대조군 | treated / control | 처치를 받은 집단 / 안 받은 집단 |
| 전 / 후 | pre / post | 처치 시점 이전 / 이후 |
| DiD | 이중차분 | (처치군 후−전) − (대조군 후−전) |
| ATT | — | 처치군에서의 평균처치효과 |
| 평행추세 | parallel trends | 처치가 없었다면 두 군이 *나란히* 변했을 것이라는 가정 |
| RDD | 회귀불연속 | 임계값(Cutoff) 근처 비교로 효과 추정 |
| PSM | 성향점수매칭 | 비슷한 특성의 대조군을 찾아 비교 |

> 💡 핵심 한 줄: **DiD = "차이의 차이".** 시간이 만든 변화(추세)와 집단이 원래 가진 차이를 *두 번 빼서* 동시에 걷어내는 발상입니다.

## 🚦 짚고 넘어가기 — Part 2 환경과 데이터

다음 질문에 답할 수 있으면 다음 Part로 넘어가세요. 틀려도 괜찮습니다.

1. 무작위 실험이 불가능한 상황의 예를 하나 들어보세요.
2. "준실험"의 핵심 과제는 무엇이라고 했나요?
3. DiD를 한 문장으로(차이의 차이) 말해보세요.

> 💡 **다음 Part 예고:** DiD로 가기 전에, *왜 단순한 비교(전후·집단)로는 안 되는지* 먼저 봅니다. 그래야 DiD가 *무엇을* 고치는지 또렷해집니다. Part 3으로 갑니다.

# Part 3. 단순 비교의 함정 — 전후 비교와 집단 비교

어떤 정책(처치)의 효과를 관찰 데이터로 보고 싶습니다. 가장 먼저 떠오르는 두 가지 단순 비교가 있는데, 둘 다 *체계적으로 틀립니다.* 왜 그런지 보면, DiD가 왜 필요한지 자연스럽게 이해됩니다.

> ❓ **이 파트에서 답할 질문:** "처치 전후 비교"와 "처치군·대조군 비교"는 각각 어떤 편향에 빠질까요?

## 💡 쉽게 말하면 — 카페 멤버십의 두 가지 착각

한 카페가 5월에 멤버십을 도입(처치)하고 매출을 봅니다. **방법 ① 전후 비교:** "도입 전(4월) 대비 후(6월) 매출이 올랐으니 멤버십 효과다!" — 그런데 6월은 원래 성수기라 멤버십이 없었어도 올랐을 수 있죠(*시간 추세*가 교란). **방법 ② 집단 비교:** "멤버십 도입한 우리 매장이 안 한 옆 매장보다 매출이 높으니 효과다!" — 그런데 우리 매장은 원래 목 좋은 자리라 *처치 전부터* 높았을 수 있습니다(*기존 차이*가 교란). 두 단순 비교 모두 숨은 교란에 속는 것입니다.

## 두 가지 단순 비교, 두 가지 편향

| 비교 방법 | 계산 | 무엇에 속나 (교란) |
|-----------|------|--------------------|
| **전후 비교** | 처치군의 (후 − 전) | *시간 추세* (처치 없어도 변했을 변화) |
| **집단 비교** | (후 시점) 처치군 − 대조군 | *기존 차이* (처치 전부터 있던 집단 차이) |

> 핵심: **전후 비교는 "시간"을 통제 못 하고, 집단 비교는 "기존 차이"를 통제 못 합니다.** DiD의 천재성은 이 둘을 *동시에* 걷어낸다는 데 있습니다. 먼저 두 편향을 데이터로 직접 봅시다.

## 예제 · 정답을 심어두고 두 단순 비교를 재보기

In [ ]:
# [C2] 합성 데이터 — 정답(ATT)을 심어두고, 두 단순 비교가 얼마나 빗나가는지 본다
np.random.seed(42)                                                # 재현성: 난수를 항상 같게 고정
n = 500
TIME_EFFECT = 12      # 시간이 만든 공통 변화 (처치와 무관) — 두 집단 모두에 적용
BASE_GAP    = 15      # 처치군이 원래 높았던 '기존 차이' (대조군보다 +15 높은 수준에서 출발)
TRUE_ATT    = 20      # 우리가 심은 진짜 처치효과

# 대조군: 전 100 → 후 100+시간효과
ctrl_pre  = np.random.normal(100, 10, n)
ctrl_post = np.random.normal(100 + TIME_EFFECT, 10, n)
# 처치군: 원래 더 높음(기존차이) → 후엔 시간효과 + 처치효과까지
trt_pre   = np.random.normal(100 + BASE_GAP, 10, n)
trt_post  = np.random.normal(100 + BASE_GAP + TIME_EFFECT + TRUE_ATT, 10, n)

naive_prepost = trt_post.mean() - trt_pre.mean()             # 전후 비교
naive_group   = trt_post.mean() - ctrl_post.mean()           # 집단 비교(후)
print(f"심은 진짜 효과(ATT)        : {TRUE_ATT}")
print(f"① 전후 비교(처치군 후-전) : {naive_prepost:5.1f}   ← 시간효과({TIME_EFFECT})만큼 부풀려짐")
print(f"② 집단 비교(후, 처치-대조): {naive_group:5.1f}   ← 기존차이({BASE_GAP})만큼 부풀려짐")

> **읽는 법:** 진짜 효과는 20인데, ① 전후 비교는 시간효과(12)가 섞여 ~32로, ② 집단 비교는 기존 차이(15)가 섞여 ~35로 부풀려집니다. 두 단순 비교 모두 *체계적으로 빗나갑니다.* 각자 "통제하지 못한 한 가지" 때문입니다 — 전후 비교는 시간을, 집단 비교는 기존 차이를.

## 그림으로 · 두 단순 비교 vs 진짜 ATT

In [ ]:
# [C3] [도식] 두 단순 비교 vs 진짜 ATT — 한눈에 빗나감을 보기
ests = {"진짜 ATT": TRUE_ATT, "전후 비교": naive_prepost, "집단 비교": naive_group}
fig, ax = plt.subplots(figsize=(7, 4))                            # 그래프 하나를 그릴 도화지 준비
ax.bar(range(3), list(ests.values()), color=["seagreen", "indianred", "orange"])
ax.axhline(TRUE_ATT, color="seagreen", ls="--", lw=1)             # 가로 기준선 표시
ax.set_xticks(range(3)); ax.set_xticklabels(list(ests.keys()))
ax.set_ylabel("estimate"); ax.set_title("Naive comparisons overshoot the true ATT")
for i, v in enumerate(ests.values()):
    ax.text(i, v + 0.5, f"{v:.1f}", ha="center", fontweight="bold")
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

> **읽는 법(그림):** 초록 점선이 진짜 효과(ATT)입니다. 전후 비교(빨강)와 집단 비교(주황) 막대가 모두 그 선을 *훌쩍 넘깁니다* — 각자 시간효과·기존차이를 효과로 오해해 부풀린 겁니다. 다음 Part의 DiD는 이 둘을 걷어내 막대를 초록 선에 닿게 합니다.

## 그림으로 · 네 칸의 평균과 반사실

In [ ]:
# [C4] [도식] 네 칸의 평균 — 두 단순 비교가 무엇을 놓치는지 한눈에
means = {"대조군 전": ctrl_pre.mean(), "대조군 후": ctrl_post.mean(),
         "처치군 전": trt_pre.mean(),  "처치군 후": trt_post.mean()}
fig, ax = plt.subplots(figsize=(8, 4.5))                          # 그래프 하나를 그릴 도화지 준비
ax.plot([0, 1], [means["대조군 전"], means["대조군 후"]], "o-", color="gray", lw=2, label="control")
ax.plot([0, 1], [means["처치군 전"], means["처치군 후"]], "o-", color="steelblue", lw=2, label="treated")
# 처치가 없었다면 처치군이 갔을 '반사실' (대조군 추세를 평행 이동)
cf = means["처치군 전"] + (means["대조군 후"] - means["대조군 전"])
ax.plot([0, 1], [means["처치군 전"], cf], "o--", color="indianred", lw=2, label="treated counterfactual")
ax.annotate("", xy=(1, means["처치군 후"]), xytext=(1, cf),
            arrowprops=dict(arrowstyle="<->", color="green", lw=2))
ax.text(1.02, (means["처치군 후"]+cf)/2, "ATT", color="green", fontweight="bold")
ax.set_xticks([0, 1]); ax.set_xticklabels(["pre", "post"]); ax.set_ylabel("outcome")
ax.set_title("Four cells: ATT = gap between treated-post and its counterfactual"); ax.legend()
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

> **읽는 법(그림):** 회색(대조군)과 파랑(처치군)이 각각 전→후로 움직입니다. 빨간 점선은 *"처치가 없었다면 처치군이 갔을 길"*(반사실) — 대조군의 추세를 처치군 출발점에 평행 이동한 것입니다. **진짜 처치효과(ATT)는 파란 점(실제 후)과 빨간 점(반사실 후)의 초록색 간격**입니다. 단순 비교들은 이 간격이 아니라 다른 길이를 재서 틀렸던 것입니다.

> 📌 **다른 산업에서는?** 이 함정은 마케팅(캠페인 전후 매출↑은 계절성일 수도), 의료(치료 후 호전은 자연회복일 수도), 교육(프로그램 참가자 성적↑은 원래 우수했을 수도) 등 *관찰 데이터로 효과를 주장하는* 모든 곳에 도사립니다.

## 🚦 짚고 넘어가기 — Part 3 단순 비교의 함정

1. 전후 비교가 통제하지 못하는 교란은 무엇인가요?
2. 집단 비교가 통제하지 못하는 교란은 무엇인가요?
3. ATT는 위 도식에서 어떤 간격으로 정의됐나요?

> 💡 **다음 Part 예고:** 두 단순 비교가 각자 한 가지를 놓친다면 — *둘을 결합*하면 어떨까요? 그게 바로 **이중차분(DiD)**의 발상입니다. Part 4로 갑니다.

# Part 4. 이중차분(DiD) — "차이의 차이"

전후 비교는 시간을 못 잡고, 집단 비교는 기존 차이를 못 잡았습니다. **이중차분(DiD)**은 두 비교를 영리하게 결합해 *두 교란을 동시에* 제거합니다. 이름 그대로 "차이를 두 번 빼는" 것입니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/d/da/Illustration_of_Difference_in_Differences.png" width="520" alt="Illustration of Difference in Differences.png">

DiD의 구조. 처치군(P)과 대조군(S)이 각각 Time 1 → Time 2로 움직입니다. **Q는 "처치가 없었다면 처치군이 갔을 길"(반사실)** 로, 대조군의 변화량만큼 P₁에서 평행 이동시켜 만듭니다. 그러면 처치효과는 **P₂ − Q**, 즉 실제 도달점과 반사실의 간격입니다.

> 🖼️ **그림 출처** — Wikimedia Commons, `Illustration of Difference in Differences.png` (CC BY-SA 3.0) · 원본·라이선스 상세: https://commons.wikimedia.org/wiki/File:Illustration_of_Difference_in_Differences.png

> ❓ **이 파트에서 답할 질문:** "차이의 차이"는 어떻게 시간 추세와 기존 차이를 *동시에* 걷어내고 진짜 효과만 남길까요?

## 💡 쉽게 말하면 — 두 화분과 비료 이야기

핵심 아이디어: **"처치군이 전→후로 변한 양"에서 "대조군이 전→후로 변한 양"을 빼자.** 대조군도 똑같은 시간(성수기·경기 등)을 겪었으니, 대조군의 변화량이 곧 *"처치가 없었어도 일어났을 변화"*의 추정치입니다. 그래서 처치군의 변화에서 그만큼을 빼면, *순수하게 처치 때문에 생긴 추가 변화*만 남습니다.

그리고 *기존 차이*는 어떻게 사라질까요? 우리는 처치군의 *수준*이 아니라 *변화량(후−전)*을 보기 때문에, 처음부터 있던 고정된 차이는 빼는 과정에서 저절로 상쇄됩니다. 한 번의 뺄셈(후−전)으로 기존 차이를, 또 한 번의 뺄셈(처치군−대조군)으로 시간 추세를 지우는 — 그래서 "이중(double)" 차분입니다.

조금 다르게 비유해볼까요. 두 화분 A(처치)와 B(대조)가 있습니다. A에만 새 비료를 줬습니다. 한 달 뒤 A가 5cm 자랐다고 "비료 효과 5cm!"라 하면 틀립니다 — 비료가 없었어도 *그냥 시간이 지나* 자랐을 테니까요. 그래서 비료를 안 준 B를 봅니다. B가 3cm 자랐다면 "시간만으로 3cm는 자란다"는 뜻입니다. 그러니 A의 5cm 중 3cm는 시간 몫이고, *비료의 순수 효과는 5−3=2cm*입니다. 이게 DiD예요 — B(대조군)의 변화로 "시간 몫"을 가늠해, A(처치군)의 변화에서 빼는 겁니다. 핵심 전제는 "비료가 없었다면 A도 B처럼 3cm 자랐을 것"(평행추세)입니다. 이 단순한 화분 비유 하나만 기억해도 DiD의 9할은 이해한 셈입니다.

## 공식 · DiD를 손으로 계산하기

$$\text{DiD} = \underbrace{(\bar{Y}^{\text{treat}}_{\text{post}} - \bar{Y}^{\text{treat}}_{\text{pre}})}_{\text{처치군 변화 (효과+시간)}} - \underbrace{(\bar{Y}^{\text{ctrl}}_{\text{post}} - \bar{Y}^{\text{ctrl}}_{\text{pre}})}_{\text{대조군 변화 (시간만)}}$$

- 첫 괄호: 처치군의 변화 = *처치효과 + 시간효과*
- 둘째 괄호: 대조군의 변화 = *시간효과* (처치 없으니)
- 둘을 빼면: **처치효과(ATT)만 남음** ✓

> 이 추정이 진짜 ATT가 되려면 결정적 가정 하나가 필요합니다 — **평행추세**(둘째 괄호가 정말 처치군의 시간효과를 대신한다는 가정). 그건 Part 6에서 본격적으로 따집니다. 먼저 계산부터 해봅시다.

In [ ]:
# [C5] DiD를 손으로 — 네 칸의 평균에서 '차이의 차이'
d_treat = trt_post.mean() - trt_pre.mean()        # 처치군 변화
d_ctrl  = ctrl_post.mean() - ctrl_pre.mean()      # 대조군 변화
did = d_treat - d_ctrl
print(f"처치군 변화 (후-전) = {d_treat:5.1f}   (처치효과 + 시간효과)")
print(f"대조군 변화 (후-전) = {d_ctrl:5.1f}   (시간효과만)")
print(f"DiD = 차이의 차이   = {did:5.1f}")
print(f"\n심은 진짜 ATT = {TRUE_ATT}  →  DiD가 거의 정확히 복원했습니다! ✓")
print("전후 비교(~32)·집단 비교(~35)와 달리, DiD는 시간·기존차이를 모두 걷어냈습니다.")

> **읽는 법:** 처치군은 전→후로 크게 변했지만(효과+시간), 대조군의 변화량을 빼주니 시간효과가 제거되고, *변화량끼리* 비교하니 기존 차이도 사라져 — 결국 진짜 ATT(20)에 거의 정확히 도달합니다. 단순 비교들이 빗나간 그 값을, DiD는 "차이의 차이" 한 번으로 복원했습니다.

## DiD를 표로 보기 — 2×2 평균과 차이의 차이

DiD가 정말 "네 칸의 차이의 차이"인지, 표로 확인하면 머리에 또렷이 남습니다.

## 예제 · 2×2 평균표와 차이의 차이

In [ ]:
# [C6] 2x2 평균표와 DiD
cell = pd.DataFrame({"pre":  [ctrl_pre.mean(),  trt_pre.mean()],
                     "post": [ctrl_post.mean(), trt_post.mean()]}, index=["control", "treated"])
cell["변화(후-전)"] = cell["post"] - cell["pre"]
print(cell.round(1))
dt, dc = cell.loc["treated", "변화(후-전)"], cell.loc["control", "변화(후-전)"]
print(f"\nDiD = 처치군 변화({dt:.1f}) - 대조군 변화({dc:.1f}) = {dt - dc:.1f}")

> **읽는 법:** 오른쪽 '변화' 열이 각 군의 전→후 변화량입니다. 처치군 변화에서 대조군 변화를 빼면(=두 변화의 차이) ATT가 나옵니다. 표의 네 칸과 두 번의 뺄셈 — 그게 DiD의 전부입니다.

## 예제 · 반사실을 명시적으로 계산하기

In [ ]:
# [C7] 반사실을 명시적으로 — '처치군이 안 받았다면' 추정값
cf = trt_pre.mean() + (ctrl_post.mean() - ctrl_pre.mean())   # 처치군 출발점 + 대조군 추세
print(f"처치군 반사실(후) 추정 = {cf:.1f}")
print(f"처치군 실제(후)        = {trt_post.mean():.1f}")
print(f"ATT = 실제 - 반사실    = {trt_post.mean() - cf:.1f}  (Part 3 도식의 초록 화살표)")

> **읽는 법:** "처치군이 안 받았다면 갔을 값(반사실)"을 *대조군의 추세*로 추정하고, 실제 처치군 값에서 빼면 ATT입니다. Part 3 도식의 초록 화살표를 숫자로 다시 본 것입니다. DiD의 네 칸 뺄셈과 정확히 같은 값이 나옵니다.

## ⌨️ 백문이 불여일타! (1) — TIME_EFFECT를 키워 DiD 복원 확인

```
[문제]
1) 위 합성 데이터에서 TIME_EFFECT(시간효과)를 12 → 30으로 키운 뒤, DiD가 여전히 진짜 ATT(20)를 복원하는지 확인해보세요.
2) 전후 비교는 얼마나 더 빗나가는지도 함께 확인해보세요.
```

In [ ]:
# [C8] ⌨️ 백문이 불여일타! (1) — TIME_EFFECT를 30으로 키워 DiD 복원 확인

# 여기에 코드를 작성하세요

<details>
<summary>(클릭) 💡 힌트</summary>

- [C2]에서 네 그룹을 만들 때 쓴 `np.random.normal(평균, 10, n)` 방식 그대로, 시간효과만 30으로 바꿔 `ctrl_post`·`trt_post`의 평균을 다시 설계해보세요.
- 전후 비교는 `trt_post.mean() - trt_pre.mean()`, DiD는 처치군 변화량에서 대조군 변화량을 빼면 됩니다 — [C2]·[C5]에서 본 계산 그대로입니다.
- 시간효과가 커질수록 전후 비교가 얼마나 더 부풀려지는지, DiD는 왜 흔들리지 않는지 비교해보세요.

</details>

## 🚦 짚고 넘어가기 — Part 4 이중차분(DiD)

1. DiD의 "두 번의 뺄셈"은 각각 어떤 교란(시간/기존차이)을 제거하나요?
2. 대조군의 변화량은 무엇을 추정하는 데 쓰이나요?
3. DiD가 진짜 ATT가 되기 위해 필요한 핵심 가정의 이름은?

> 💡 **다음 Part 예고:** 네 칸 평균을 손으로 빼는 대신, **회귀 한 줄로 DiD를 추정**하고 신뢰구간·p값까지 얻는 법을 Part 5에서 배웁니다.

# Part 5. DiD를 회귀로 구현하기

손으로 네 칸을 빼도 되지만, **회귀(OLS)로 DiD를 추정하면** 한 줄로 끝나고 신뢰구간·p값·공변량 보정까지 덤으로 얻습니다. 앞서 배운 회귀가 인과추론 도구로 변신하는 순간이에요(회귀의 "세 번째 얼굴"이 여기 있었습니다).

> ❓ **이 파트에서 답할 질문:** DiD를 회귀식으로 어떻게 표현하고, 어느 계수가 ATT일까요?

## 💡 쉽게 말하면 — 상호작용항이 곧 DiD

데이터를 "긴(Long) 형태"로 만들어, 각 관측치에 두 개의 0/1 변수를 답니다 — `treat`(처치군이면 1)와 `post`(처치 후면 1). 그리고 다음 회귀를 적합합니다.

$$Y = \beta_0 + \beta_1\,\text{treat} + \beta_2\,\text{post} + \beta_3\,(\text{treat}\times\text{post}) + \varepsilon$$

- $\beta_1$(treat): 처치군의 *기존 차이* (전 시점 집단 차이)
- $\beta_2$(post): *시간 추세* (대조군이 전→후로 변한 양)
- $\beta_3$(treat×post): **바로 이 상호작용 계수가 DiD = ATT** ✓

상호작용항이 "처치군이면서 동시에 후 시점일 때만" 켜지므로, 정확히 *처치가 더해진 추가 효과*를 집어냅니다. 네 칸 뺄셈과 수학적으로 동일하지만, 회귀라 표준오차·신뢰구간이 공짜로 나와요.

## 예제 · long 형태로 만들고 DiD 회귀 적합

In [ ]:
# [C9] 데이터를 long 형태로 만들고 DiD 회귀 적합
df = pd.DataFrame({
    "y":     np.concatenate([ctrl_pre, ctrl_post, trt_pre, trt_post]),
    "treat": np.r_[np.zeros(2*n), np.ones(2*n)],
    "post":  np.r_[np.zeros(n), np.ones(n), np.zeros(n), np.ones(n)],
})
did_model = smf.ols("y ~ treat + post + treat:post", data=df).fit()  # DiD 회귀 — treat:post 상호작용 계수가 곧 ATT
print(did_model.summary().tables[1])                              # 회귀 결과에서 '계수 표'만 출력
att = did_model.params["treat:post"]
ci = did_model.conf_int().loc["treat:post"]                       # 계수의 95% 신뢰구간
print(f"\nDiD(ATT) 추정 = {att:.2f},  95% CI = [{ci[0]:.2f}, {ci[1]:.2f}]")
print(f"심은 진짜 ATT = {TRUE_ATT}  → 회귀의 treat:post 계수가 ATT를 복원! ✓")

> **읽는 법:** `treat:post`의 `coef`가 DiD 추정치(≈20=ATT)이고, `P>|t|`로 유의성을, `[0.025 0.975]`로 신뢰구간을 얻습니다. `treat` 계수는 기존 차이(~15), `post` 계수는 시간 추세(~12)를 각각 보여줍니다 — 회귀 한 줄이 네 칸의 모든 이야기를 분해해 줍니다. 게다가 신뢰구간이 0을 포함하지 않으니 "효과가 0이 아니다"라고 말할 수 있습니다.

## 그림으로 · 회귀가 분해한 세 조각

In [ ]:
# [C10] 회귀가 분해한 세 조각을 시각화 — 기존차이 / 시간추세 / 처치효과(ATT)
b = did_model.params
parts = {"treat\n(기존 차이)": b["treat"], "post\n(시간 추세)": b["post"], "treat:post\n(ATT)": b["treat:post"]}
fig, ax = plt.subplots(figsize=(7, 4))                            # 그래프 하나를 그릴 도화지 준비
colors = ["gray", "orange", "seagreen"]
ax.bar(range(3), list(parts.values()), color=colors)
ax.set_xticks(range(3)); ax.set_xticklabels(list(parts.keys()))
ax.axhline(0, color="black", lw=0.8); ax.set_ylabel("coefficient")  # 가로 기준선 표시
ax.set_title("DiD regression decomposes the four cells")
for i, v in enumerate(parts.values()):
    ax.text(i, v+0.5, f"{v:.1f}", ha="center", fontweight="bold")
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

> **읽는 법(그림):** 회귀가 결과의 변동을 세 조각으로 나눠 보여줍니다 — 처치군의 *기존 차이*(회색), *시간 추세*(주황), 그리고 우리가 원하던 *순수 처치효과 ATT*(초록). 단순 비교가 이 조각들을 뒤섞어 틀렸다면, DiD 회귀는 깔끔히 분리해 ATT만 집어냅니다.

## 공변량을 더하면 — 더 정밀하게

회귀 형태의 또 다른 장점: **공변량(통제 변수)을 추가**해 추정을 더 정밀하게(또는 일부 교란을 추가 보정) 만들 수 있습니다. 식에 `+ x` 를 붙이기만 하면 됩니다. 단, 공변량은 *처치의 영향을 받지 않는* 사전 변수여야 합니다(처치 후 변수를 넣으면 오히려 편향).

In [ ]:
# [C11] 공변량 추가 DiD — 사전 특성(예: 매장 규모)을 통제하면 표준오차가 줄 수 있다
np.random.seed(1)                                                 # 재현성: 난수를 항상 같게 고정
df["x_size"] = np.random.normal(50, 10, len(df)) + 3*df["treat"]   # 처치군이 약간 큰 매장
df["y2"] = df["y"] + 0.4*(df["x_size"] - 50)                       # 규모가 결과에 약간 영향
m_cov = smf.ols("y2 ~ treat + post + treat:post + x_size", data=df).fit()  # DiD 회귀 — treat:post 상호작용 계수가 곧 ATT
print(f"공변량 없는 ATT 추정 표준오차 : {smf.ols('y2 ~ treat + post + treat:post', data=df).fit().bse['treat:post']:.3f}")  # DiD 회귀 — treat:post 상호작용 계수가 곧 ATT
print(f"공변량 포함 ATT 추정 표준오차 : {m_cov.bse['treat:post']:.3f}")
print(f"공변량 포함 ATT = {m_cov.params['treat:post']:.2f} (95% CI 폭이 좁아지면 더 정밀)")

> **읽는 법:** 결과를 설명하는 사전 공변량을 넣으면 잔차 분산이 줄어, ATT 추정의 표준오차가 작아질 수 있습니다(더 정밀한 추정). 다만 공변량 선택은 신중해야 해요 — *처치 이후에 측정된* 변수나 *처치의 결과인* 변수를 넣으면 오히려 편향을 부릅니다(앞서 배운 "나쁜 통제" 문제).

## 회귀가 네 칸을 그대로 재현한다 + 표준오차 이야기

DiD 회귀가 정말 네 칸 평균과 같은 걸 계산하는지 예측값으로 확인하고, 실무 DiD에서 빠지지 않는 *표준오차* 주의점도 짚습니다.

## 확인 · 회귀 예측값 = 네 칸 평균

In [ ]:
# [C12] 회귀 예측값 = 네 칸 평균 (회귀가 DiD 모형을 그대로 적합)
df["yhat"] = did_model.predict(df)
print("회귀가 예측한 네 칸 평균:")
print(df.groupby(["treat", "post"])["yhat"].mean().round(1))      # 집단별로 묶어서 계산

> **읽는 법:** 회귀 예측값을 네 칸으로 평균 내면 원래 데이터의 네 칸 평균과 정확히 일치합니다. DiD 회귀가 "네 칸 모형"을 그대로 적합하고 있다는 증거입니다.

## 확인 · 언제 robust/clustered SE가 달라지나

In [ ]:
# [C13] 표준오차 — 언제 robust/clustered SE가 달라지는가
m_hc = smf.ols("y ~ treat + post + treat:post", data=df).fit(cov_type="HC1")  # 이분산에 강한(robust) 표준오차 — 계수는 그대로, SE만 보정
print("[네 칸 크기가 같은 경우]")
print(f"  기본 SE        : {did_model.bse['treat:post']:.3f}")
print(f"  robust(HC1) SE : {m_hc.bse['treat:post']:.3f}")
print("  → 두 값이 같습니다. 네 칸의 표본 수가 동일한 '포화(saturated) 2x2 DiD'에서는")
print("     기본 SE와 robust SE가 대수적으로 일치하기 때문입니다(우연이 아닙니다).")

# 칸 크기가 다르고 이분산까지 있으면 이야기가 달라집니다.
np.random.seed(7)                                                 # 재현성: 난수를 항상 같게 고정
rows = []
for tr, po, nn, sd in [(0, 0, 500, 10), (0, 1, 500, 10), (1, 0, 500, 10), (1, 1, 150, 30)]:
    mu = 100 + BASE_GAP*tr + TIME_EFFECT*po + TRUE_ATT*tr*po
    rows.append(pd.DataFrame({"treat": tr, "post": po, "y": np.random.normal(mu, sd, nn)}))
unb = pd.concat(rows)     # 처치군-사후 칸만 표본이 적고(150) 분산이 큼(sd=30)
u_ols = smf.ols("y ~ treat + post + treat:post", data=unb).fit()  # DiD 회귀 — treat:post 상호작용 계수가 곧 ATT
u_hc  = smf.ols("y ~ treat + post + treat:post", data=unb).fit(cov_type="HC1")  # 이분산에 강한(robust) 표준오차 — 계수는 그대로, SE만 보정
print("\n[칸 크기가 다르고 이분산이 있는 경우]")
print(f"  기본 SE        : {u_ols.bse['treat:post']:.3f}")
print(f"  robust(HC1) SE : {u_hc.bse['treat:post']:.3f}   ({u_hc.bse['treat:post']/u_ols.bse['treat:post']:.1f}배)")
print("  → 기본 SE가 불확실성을 과소평가합니다(신뢰구간이 실제보다 좁아져 '과신').")

print("\n→ 실제 패널 DiD에서는 같은 단위(개인·지역)가 반복 관측되므로, 보통 단위로 '클러스터 표준오차'를 씁니다.")

> **읽는 법:** 계수(ATT)는 그대로지만 *표준오차*는 어떤 가정을 두느냐에 따라 달라집니다. 다만 **네 칸의 표본 수가 같은 기본형 2×2 DiD에서는 기본 SE와 robust SE가 정확히 일치**합니다 — 그래서 첫 번째 출력에서는 차이가 없습니다. 칸 크기가 다르거나 이분산이 있으면(두 번째 출력) robust SE가 커지고, 기본 SE를 그대로 믿으면 신뢰구간이 실제보다 좁아져 *과신*하게 됩니다.

> 그리고 같은 단위(개인·지역)가 여러 시점에 반복 관측되는 *패널*이라면, 단위별로 묶는 **클러스터 표준오차**가 표준입니다. 계수만큼 SE도 중요합니다 — SE를 잘못 잡으면 p값과 신뢰구간이 함께 틀어집니다.

## 실제 정책으로 DiD 해보기 — 켄터키주 산재보험 개정 (`injury`)

이제 진짜 데이터로 같은 절차를 밟아봅니다. 합성 데이터와 달리 **정답을 모른 채** 추정해야 합니다.

**배경.** 1980년 7월, 미국 켄터키주는 산업재해 근로자에게 지급하는 **주간 급여의 상한(cap)** 을 크게 올렸습니다. 그런데 이 인상은 **원래 소득이 높았던 근로자에게만** 실질적인 혜택이 됐습니다 — 저소득 근로자는 애초에 상한에 걸리지 않았으니까요.

그래서 자연스럽게 DiD 구조가 만들어집니다.

| DiD 요소 | 이 정책에서는 | 변수 |
| --- | --- | --- |
| **처치군** | 고소득 재해 근로자(상한 인상의 혜택을 받음) | `highearn = 1` |
| **대조군** | 저소득 재해 근로자(정책과 무관) | `highearn = 0` |
| **전/후** | 1980년 7월 정책 시행 전 / 후 | `afchnge = 0 / 1` |
| **결과** | 산재 휴업 기간(주) — 로그 변환값 | `ldurat = log(durat)` |

**연구 질문:** 급여가 늘어나면 재해 근로자가 **더 오래 쉬게 될까?** (경제학에서 말하는 *도덕적 위험*의 검증)

> 📖 **왜 결과를 로그로 변환하나요?** 휴업 기간은 강하게 우편향된 변수입니다(대부분 짧고 일부 매우 김 — D012에서 배운 그 모양). 로그를 씌우면 분포가 대칭에 가까워지고, **계수를 "몇 % 변화"로 읽을 수 있어** 해석이 쉬워집니다. 로그 종속변수의 계수 $\beta$는 대략 $100\beta$%의 변화를 뜻하며, 정확히는 $(e^{\beta}-1)\times 100$%입니다.

## 예제 · 켄터키 산재보험 DiD 회귀

In [ ]:
# [C13b] 실제 정책 DiD — 켄터키주 산재보험 급여 상한 인상 (1980)
ky = injury[injury["ky"] == 1].copy()        # 켄터키주만 사용

# [1] 먼저 네 칸의 평균을 눈으로 확인 (DiD의 출발점)
cells = ky.groupby(["highearn", "afchnge"])["durat"].agg(["count", "mean"]).round(2)  # 집단별로 묶어서 계산
cells.index = pd.MultiIndex.from_tuples(
    [("저소득(대조)", "정책 전"), ("저소득(대조)", "정책 후"),
     ("고소득(처치)", "정책 전"), ("고소득(처치)", "정책 후")])
print("[네 칸의 평균 휴업 기간(주)]")
print(cells)

d_treat = cells.loc[("고소득(처치)", "정책 후"), "mean"] - cells.loc[("고소득(처치)", "정책 전"), "mean"]
d_ctrl  = cells.loc[("저소득(대조)", "정책 후"), "mean"] - cells.loc[("저소득(대조)", "정책 전"), "mean"]
print(f"\n손으로 계산한 DiD = {d_treat:.2f} - {d_ctrl:.2f} = {d_treat - d_ctrl:+.2f}주")

# [2] 회귀로 추정 — 결과는 로그 휴업 기간, 표준오차는 이분산에 강한 HC1
m_ky = smf.ols("ldurat ~ afchnge + highearn + afchnge:highearn", data=ky).fit(cov_type="HC1")  # 이분산에 강한(robust) 표준오차 — 계수는 그대로, SE만 보정
print("\n[DiD 회귀 — 종속변수 = log(휴업 주)]")
print(m_ky.summary().tables[1])                                   # 회귀 결과에서 '계수 표'만 출력

att_log = m_ky.params["afchnge:highearn"]
ci_log = m_ky.conf_int().loc["afchnge:highearn"]                  # 계수의 95% 신뢰구간
print(f"\nATT(로그 스케일) = {att_log:+.4f}   95% CI [{ci_log[0]:+.4f}, {ci_log[1]:+.4f}]")
print(f"→ % 해석: 휴업 기간이 약 {(np.exp(att_log)-1)*100:+.1f}% 늘어남"
      f"  (95% CI {(np.exp(ci_log[0])-1)*100:+.1f}% ~ {(np.exp(ci_log[1])-1)*100:+.1f}%)")
print(f"   p = {m_ky.pvalues['afchnge:highearn']:.4f}")

> **읽는 법:** `afchnge:highearn` 계수가 약 **+0.19**입니다. 로그 종속변수이므로 **"정책 후 고소득 재해 근로자의 휴업 기간이 대조군 대비 약 21% 길어졌다"** 로 읽습니다(95% 신뢰구간도 0을 포함하지 않습니다).
>
> 나머지 두 계수도 의미가 있습니다.
> - `highearn`(+0.26) — 정책 전에도 고소득 근로자의 휴업이 더 길었습니다(**기존 차이**).
> - `afchnge`(+0.01, 비유의) — 저소득군의 휴업 기간은 정책 전후로 거의 안 변했습니다(**시간 추세**가 거의 0).
>
> 즉 "고소득군만 정책 후에 추가로 길어졌다"가 데이터에 남았고, 그 추가분이 ATT입니다. 이것이 Meyer, Viscusi & Durbin(1995)의 유명한 결과이며, 급여가 늘면 복귀가 늦어진다는 *도덕적 위험*의 근거로 인용됩니다.
>
> ⚠️ **그래도 가정은 남아 있습니다.** 이 추정이 인과이려면 "정책이 없었다면 두 소득군의 휴업 기간이 나란히 움직였을 것"(평행추세)이 성립해야 합니다. 이 데이터는 전·후 각 한 시점뿐이라 **처치 전 추세를 그려볼 수가 없습니다** — DiD의 가장 흔한 현실적 제약입니다. 그래서 원 논문은 *다른 주(미시간)* 를 이용한 비교 등 여러 보강을 함께 제시합니다. 다음 Part에서 여러 시점이 있을 때의 점검법을 배웁니다.
>
> 💡 **표준오차 메모:** `.fit(cov_type="HC1")`로 이분산에 강한 표준오차를 썼습니다. 개인 단위 관측이고 같은 개인이 반복 등장하지 않으므로 클러스터링은 하지 않았습니다. 반면 *같은 주·같은 기업이 여러 시점 반복*되는 패널이라면 그 단위로 클러스터 표준오차를 쓰는 것이 표준입니다.

## ⌨️ 백문이 불여일타! (2) — did_model의 treat·post 계수 비교

```
[문제]
1) 위 did_model에서 post 계수와 treat 계수를 읽어보세요.
2) 각각이 우리가 심은 TIME_EFFECT(12)·BASE_GAP(15)과 비슷한지 확인해보세요.
```

In [ ]:
# [C14] ⌨️ 백문이 불여일타! (2) — did_model의 treat·post 계수를 심은 값과 비교

# 여기에 코드를 작성하세요

<details>
<summary>(클릭) 💡 힌트</summary>

- `did_model.params`를 출력하면 각 항의 계수만 뽑아볼 수 있습니다.
- `treat`·`post`·`treat:post` 세 계수를 우리가 심은 `BASE_GAP`(15)·`TIME_EFFECT`(12)·ATT(20)와 하나씩 짝지어보세요.
- [C9]의 회귀 결과 표에서 본 `coef` 열과 같은 값입니다.

</details>

## 🚦 짚고 넘어가기 — Part 5 DiD 회귀

1. DiD 회귀에서 어느 계수가 ATT인가요? 왜 그 항인가요?
2. `treat` 계수와 `post` 계수는 각각 무엇을 나타내나요?
3. 공변량을 추가할 때 주의할 점(어떤 변수를 넣으면 안 되나)은?

> 💡 **다음 Part 예고:** DiD는 강력하지만, *단 하나의 가정* 위에 서 있습니다 — **평행추세**. 이 가정이 깨지면 DiD는 완전히 틀려요. 어떻게 점검하는지 Part 6에서 배웁니다.

# Part 6. 평행추세 — DiD의 생명

DiD 전체가 기대는 가정이 **평행추세(Parallel Trends)**입니다. "처치가 *없었다면*, 처치군과 대조군이 *나란히(평행하게)* 변했을 것"이라는 가정입니다. 이게 성립해야 "대조군의 변화 = 처치군의 시간효과"라는 DiD의 핵심 대체가 정당해집니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/f/fb/Parallel_Trend_Assumption.png" width="560" alt="Parallel Trend Assumption.png">

평행추세 가정. 파란 실선이 처치군의 **관측된** 경로, 파란 점선이 **가정된 반사실**(처치가 없었다면 대조군과 나란히 갔을 경로)입니다. 처치효과는 이 둘의 간격이며, **점선은 데이터가 아니라 가정**이라는 점이 핵심입니다.

> 🖼️ **그림 출처** — Wikimedia Commons, `Parallel Trend Assumption.png` (CC BY-SA 3.0) · 원본·라이선스 상세: https://commons.wikimedia.org/wiki/File:Parallel_Trend_Assumption.png

> ❓ **이 파트에서 답할 질문:** 평행추세 가정은 정확히 무엇이고, *관측 가능한* 데이터로 어떻게 그럴듯함을 점검할까요?

## 💡 쉽게 말하면 — 점검할 수 있는 것과 없는 것

평행추세는 *반사실*에 대한 가정이라 **직접 증명할 수는 없습니다**("처치가 없었다면"은 영원히 못 보니까요). 하지만 *간접 증거*는 모을 수 있습니다. 핵심 아이디어: **처치 *이전* 여러 시점에서 두 군이 실제로 나란히 움직였다면, 처치가 없었어도 계속 나란했으리라 믿을 만하다.** 그래서 처치 전 여러 기간의 추세를 그려, 두 선이 평행한지 눈으로 확인합니다. 이것이 DiD를 쓸 때 *반드시* 보여줘야 하는 그림입니다.

## 예제 · 평행추세 점검용 패널 데이터 만들기

In [ ]:
# [C15] 평행추세 점검용 패널 데이터 — 처치 전 여러 시점 포함 (평행추세 '성립' 케이스)
# (시드 29: 처치 전 기울기가 구조적으로 같은 이 데이터에서 뒤의 진단 검정들이
#  '넉넉히 비유의'하게 나오도록 골랐습니다 — 우연히 p가 0.05 근처로 떨어지면 시범 예시로 혼란스럽기 때문)
np.random.seed(29)                                                # 재현성: 난수를 항상 같게 고정
times = np.array([-4, -3, -2, -1, 0, 1, 2])     # 0부터 처치 시작 (post)
def panel(slope, base, att, treat):
    rows = []
    for t in times:
        eff = att if (treat and t >= 0) else 0
        y = base + slope*t + eff + np.random.normal(0, 4, 300)
        rows.append(pd.DataFrame({"time": t, "treat": int(treat), "y": y}))
    return pd.concat(rows)

# 두 군의 시간 기울기를 같게(=평행) 두고, 처치군에만 t>=0에서 ATT 점프
good = pd.concat([panel(slope=5, base=100, att=0,  treat=False),
                  panel(slope=5, base=80,  att=22, treat=True)])
trend = good.groupby(["time", "treat"])["y"].mean().unstack()     # 집단별로 묶어서 계산
print(trend.round(1))

## 그림으로 · 평행추세가 성립하는 경우

In [ ]:
# [C16] [도식] 평행추세 — 처치 전 두 선이 나란하면 DiD를 믿을 만하다
fig, ax = plt.subplots(figsize=(9, 4.5))                          # 그래프 하나를 그릴 도화지 준비
ax.plot(trend.index, trend[0], "o-", color="gray", label="control")
ax.plot(trend.index, trend[1], "o-", color="steelblue", label="treated")
ax.axvline(-0.5, color="indianred", ls="--", label="treatment starts")  # 세로 기준선 표시
ax.set_xlabel("time (0 = treatment)"); ax.set_ylabel("mean outcome")
ax.set_title("Parallel pre-trends (good): lines move together before treatment"); ax.legend()
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

> **읽는 법(그림):** 빨간 점선(처치 시작) *왼쪽*에서 두 선이 **나란히** 올라갑니다 — 처치 전 평행추세가 성립한다는 증거입니다. 그러다 처치 후(오른쪽) 처치군(파랑)만 위로 점프합니다. 이 점프가 ATT예요. 처치 전이 평행했으니, "처치 없었으면 계속 평행했을 것"이라 믿고 DiD를 정당하게 적용할 수 있습니다.

## 반례 · 처치 전 추세가 다르면 DiD는 틀린다

In [ ]:
# [C17] 반례 — 처치 전부터 두 군의 추세가 '다르면' DiD는 틀린다
np.random.seed(42)                                                # 재현성: 난수를 항상 같게 고정
bad = pd.concat([panel(slope=5, base=100, att=0,  treat=False),
                 panel(slope=9, base=80,  att=0,  treat=True)])   # 처치군이 원래 더 가파름! ATT=0인데도
trend_bad = bad.groupby(["time", "treat"])["y"].mean().unstack()  # 집단별로 묶어서 계산
fig, ax = plt.subplots(figsize=(9, 4.5))                          # 그래프 하나를 그릴 도화지 준비
ax.plot(trend_bad.index, trend_bad[0], "o-", color="gray", label="control")
ax.plot(trend_bad.index, trend_bad[1], "o-", color="steelblue", label="treated (steeper pre-trend)")
ax.axvline(-0.5, color="indianred", ls="--")                      # 세로 기준선 표시
ax.set_title("Non-parallel pre-trends (bad): DiD would falsely 'find' an effect")
ax.set_xlabel("time (0 = treatment)"); ax.set_ylabel("mean outcome"); ax.legend()
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

# 이 데이터엔 진짜 ATT=0인데, DiD는 가짜 효과를 보고한다
bad["post"] = (bad["time"] >= 0).astype(int)
fake = smf.ols("y ~ treat + post + treat:post", data=bad).fit().params["treat:post"]  # DiD 회귀 — treat:post 상호작용 계수가 곧 ATT
print(f"진짜 ATT = 0 인데, DiD 추정 = {fake:.1f}  ← 평행추세 위반이 만든 '가짜 효과'!")

> **읽는 법(그림):** 여기서는 처치 *전부터* 처치군(파랑)이 대조군보다 *가파르게* 오릅니다 — 평행추세가 깨진 겁니다. 이 데이터엔 진짜 효과가 **0**인데도, DiD는 "효과가 있다"는 *가짜 결론*을 냅니다. 처치군이 원래 더 빨리 성장하던 흐름을 DiD가 "처치효과"로 오해한 것입니다. **그래서 DiD를 쓰기 전에 반드시 처치 전 추세를 그려봐야 합니다.**

## 예제 · 플라시보 검정 (가짜 처치 시점)

In [ ]:
# [C18] 플라시보(placebo) 검정 — '가짜 처치 시점'에선 효과가 없어야 정상
# 처치 전 기간만 떼어, 가짜로 t=-2를 '처치 시작'이라 치고 DiD를 돌린다
pre_only = good[good["time"] < 0].copy()
pre_only["fake_post"] = (pre_only["time"] >= -2).astype(int)
placebo = smf.ols("y ~ treat + fake_post + treat:fake_post", data=pre_only).fit()
pe = placebo.params["treat:fake_post"]; pp = placebo.pvalues["treat:fake_post"]
print(f"플라시보 DiD(가짜 처치) = {pe:.2f}, p = {pp:.3f}")
print("→ 0에 가깝고 유의하지 않으면(p 큼) 평행추세 가정이 그럴듯하다는 방증입니다.")

> **읽는 법:** 플라시보 검정은 *실제 처치가 없던 기간*에 "가짜 처치 시점"을 넣고 DiD를 돌립니다. 거기서 효과가 0에 가깝게 나오면(유의하지 않으면), "우리 DiD가 잡은 진짜 처치 후 효과는 평행추세 위반이 아니라 진짜다"라는 강력한 방증이 됩니다. 인과 분석가가 *스스로의 가정을 공격해보는* 정직한 습관입니다.

> 📌 **다른 산업에서는?** 평행추세 점검·플라시보 검정은 정책평가(최저임금·금연정책 효과), 마케팅(지역별 캠페인 효과), 플랫폼(기능 출시의 지역 롤아웃 효과) 분석에서 결과를 신뢰받기 위한 *필수 절차*입니다.

## 비교 · 평행추세 성립 vs 위반에서의 DiD

In [ ]:
# [C19] 평행추세 성립(good) vs 위반(bad)에서 DiD 추정 비교 — 가정이 전부다
good["post"] = (good["time"] >= 0).astype(int)
att_good = smf.ols("y ~ treat + post + treat:post", data=good).fit().params["treat:post"]  # DiD 회귀 — treat:post 상호작용 계수가 곧 ATT
att_bad  = smf.ols("y ~ treat + post + treat:post", data=bad).fit().params["treat:post"]  # DiD 회귀 — treat:post 상호작용 계수가 곧 ATT
print(f"평행추세 성립(good): DiD ATT = {att_good:.1f}   (진짜 효과 22에 근접 ✓)")
print(f"평행추세 위반(bad) : DiD ATT = {att_bad:.1f}   (진짜 효과 0인데 가짜 효과! ✗)")
print("→ 같은 DiD 공식도 평행추세 가정이 깨지면 완전히 틀립니다.")

> **읽는 법:** 똑같은 DiD 회귀인데, 평행추세가 성립한 데이터에선 진짜 효과(22)를 잘 잡고, 위반한 데이터에선 진짜가 0인데도 큰 가짜 효과를 냅니다. **DiD의 정확성은 공식이 아니라 가정에 달려 있어요** — 오늘의 핵심을 숫자로 못박는 비교입니다. 그래서 결과를 내기 전에 *반드시* 평행추세부터 확인하는 겁니다.

## 이벤트 스터디 — 시점별로 평행추세와 효과를 한눈에

평행추세 점검과 효과 추정을 *한 그림*으로 합치는 방법이 **이벤트 스터디(Event Study)**입니다. 각 시점에서 "처치군−대조군 격차"를 처치 *직전(t=−1)* 기준으로 그려요. 처치 전 점들이 0 근처에 평평하면 평행추세, 처치 후 점들이 위로 뛰면 그게 효과(동적 ATT)입니다.

## 예제 · 이벤트 스터디 (시점별 격차)

In [ ]:
# [C20] 이벤트 스터디 — 시점별 (처치군 - 대조군) 격차, t=-1 기준 상대화
es = good.groupby(["time", "treat"])["y"].mean().unstack()        # 집단별로 묶어서 계산
es["gap"] = es[1] - es[0]
es["rel"] = es["gap"] - es["gap"].loc[-1]      # t=-1을 0으로
fig, ax = plt.subplots(figsize=(9, 4))                            # 그래프 하나를 그릴 도화지 준비
ax.plot(es.index, es["rel"], "o-", color="purple")
ax.axhline(0, color="gray"); ax.axvline(-0.5, color="indianred", ls="--", label="treatment")  # 세로 기준선 표시
ax.set_xlabel("time (0 = treatment)"); ax.set_ylabel("treated - control (rel. to t=-1)")
ax.set_title("Event study: flat before, jumps after"); ax.legend()
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

> **읽는 법(그림):** 처치 전(−3,−2,−1) 점들이 0 근처에 *평평*합니다 — 평행추세의 시각적 증거입니다. 처치 후 점들이 위로 뛰고, 그 높이가 시점별 효과(동적 ATT)입니다. 이벤트 스터디 하나로 *가정 점검*과 *효과 추정*을 동시에 보여줄 수 있어, 실무 DiD 보고의 단골 그림입니다.

## 확인 · 처치 전 추세 차이의 형식적 검정

In [ ]:
# [C21] 평행추세의 '형식적' 점검 — 처치 전 기간에서 추세 차이가 유의한가
pre_g = good[good["time"] < 0]
m_pt = smf.ols("y ~ treat * time", data=pre_g).fit()
print(f"처치 전 treat:time 계수(두 군의 추세 차이) = {m_pt.params['treat:time']:.3f}, p = {m_pt.pvalues['treat:time']:.3f}")
print("→ 0에 가깝고 비유의면(p 큼) '처치 전 추세가 평행'이라는 가정을 지지합니다.")

> **읽는 법:** 처치 *전* 데이터만으로 "두 군의 시간 기울기가 다른가"(treat×time)를 검정합니다. 계수가 0에 가깝고 유의하지 않으면 평행추세를 통계적으로도 지지하는 셈입니다. 단, 이 검정을 *통과해도* 평행추세를 *증명*한 건 아니라는 점(검정력 한계)은 늘 기억하세요 — 어디까지나 *방증*입니다.

## 실데이터로 평행추세 보기 — 캘리포니아 담배세 인상 (`Cigar`)

합성 데이터에서는 평행추세가 성립하는지 우리가 알고 있었습니다. 실데이터에서는 어떨까요? 여러 시점이 있는 **패널 데이터**로 직접 그려보고 검정해봅시다. **결론을 미리 말하면 — 가정이 생각만큼 잘 성립하지 않습니다.** 그때 분석가가 무엇을 해야 하는지가 이 절의 핵심입니다.

**배경.** 1988년 11월 캘리포니아는 주민투표로 **Proposition 99**를 통과시켜 담뱃세를 대폭 인상했습니다(1989년 1월 시행). 담배 판매량이 정말 줄었을까요?

- **처치군:** 캘리포니아 1개 주
- **대조군:** 나머지 45개 주(세금 인상 없음)
- **결과:** 1인당 담배 판매량(갑)
- **처치 시점:** 1989년

이 사례는 인과추론 문헌에서 **합성통제법(Synthetic Control)** 의 대표 예제로 유명합니다(Abadie, Diamond & Hainmueller, 2010).

## 예제 · 캘리포니아 담배세 이벤트 스터디

In [ ]:
# [C21b] 실데이터 이벤트 스터디 — 캘리포니아(주 코드 5) vs 나머지 45개 주
CA = 5                                        # Cigar 데이터에서 캘리포니아의 주 코드
cig = cigar[cigar["year"] >= 75].copy()       # 1975년 이후만 사용
cig["treat"] = (cig["state"] == CA).astype(int)
cig["post"] = (cig["year"] >= 89).astype(int)
cig["yr"] = 1900 + cig["year"]

# 연도별 평균 판매량: 캘리포니아 vs 나머지 주 평균
trend_ca = cig.groupby(["yr", "treat"])["sales"].mean().unstack()  # 집단별로 묶어서 계산
trend_ca.columns = ["대조군(45개 주 평균)", "캘리포니아"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))                 # 그림 한 장에 그래프 2개(1행 2열)

# (왼쪽) 원자료 추세 — 처치 전 두 선이 나란한가?
axes[0].plot(trend_ca.index, trend_ca["대조군(45개 주 평균)"], "o-", color="gray", ms=4, label="control (45 states)")
axes[0].plot(trend_ca.index, trend_ca["캘리포니아"], "o-", color="steelblue", ms=4, label="California")
axes[0].axvline(1988.5, color="indianred", ls="--", label="Prop 99 (1989)")  # 세로 기준선 표시
axes[0].set_xlabel("year"); axes[0].set_ylabel("packs per capita")
axes[0].set_title("Raw trends: parallel before 1989?"); axes[0].legend(fontsize=8)

# (오른쪽) 이벤트 스터디 — 격차를 처치 직전(1988) 기준으로 상대화
gap = trend_ca["캘리포니아"] - trend_ca["대조군(45개 주 평균)"]
rel = gap - gap.loc[1988]
axes[1].plot(rel.index, rel.values, "o-", color="purple", ms=4)
axes[1].axhline(0, color="gray"); axes[1].axvline(1988.5, color="indianred", ls="--")  # 세로 기준선 표시
axes[1].set_xlabel("year"); axes[1].set_ylabel("CA − control  (rel. to 1988)")
axes[1].set_title("Event study: flat before, drops after")
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

# 처치 전 추세 차이의 형식적 점검 (1975~1988)
pre_ca = cig[cig["year"] <= 88]
m_pre = smf.ols("sales ~ treat * yr", data=pre_ca).fit(cov_type="HC1")  # 이분산에 강한(robust) 표준오차 — 계수는 그대로, SE만 보정
print(f"[처치 전(1975~1988) 추세 차이] treat:yr = {m_pre.params['treat:yr']:+.3f}"
      f"  (p = {m_pre.pvalues['treat:yr']:.3f})  → 0에 가깝고 비유의면 평행추세 지지")

# 참고: 2x2 DiD 추정치 (처치 전 평균 vs 처치 후 평균)
m_did_ca = smf.ols("sales ~ treat + post + treat:post", data=cig).fit(cov_type="HC1")  # 이분산에 강한(robust) 표준오차 — 계수는 그대로, SE만 보정
print(f"[2x2 DiD 추정] ATT = {m_did_ca.params['treat:post']:+.2f} 갑/인")
print(f"  1988년 캘리포니아 {trend_ca.loc[1988,'캘리포니아']:.1f} → 1992년 {trend_ca.loc[1992,'캘리포니아']:.1f} 갑")
print(f"  같은 기간 대조군  {trend_ca.loc[1988,'대조군(45개 주 평균)']:.1f} → {trend_ca.loc[1992,'대조군(45개 주 평균)']:.1f} 갑")

> **읽는 법(그림):** 왼쪽 그림만 보면 1989년 이전 두 선이 *그럭저럭 나란히* 하락하는 것처럼 보입니다. 그리고 1989년 이후 캘리포니아만 더 빠르게 떨어집니다. 2×2 DiD 추정치는 약 **−14갑/인**입니다.
>
> **그런데 형식적 점검이 경고를 보냅니다.** 처치 전(1975~1988) 추세 차이 검정에서 `treat:yr` 계수가 약 **−1.06이고 p ≈ 0.002** — *처치 전에도 캘리포니아가 이미 더 빠르게 감소하고 있었다*는 뜻입니다. **평행추세 가정이 통계적으로 기각됩니다.**
>
> 오른쪽 이벤트 스터디를 다시 보면 그 신호가 실제로 보입니다 — 1989년 이전 구간이 완전히 평평하지 않고 이미 완만하게 아래로 기울어 있습니다. 즉 **DiD가 잡은 −14갑 중 일부는 정책 효과가 아니라 원래 진행 중이던 추세**일 수 있습니다.
>
> 🎯 **이것이 이 절의 진짜 교훈입니다.** 그림만 보고 "나란해 보인다"로 넘어가면 안 되고, **검정으로 확인해야** 합니다. 그리고 가정이 흔들리면 결과를 밀어붙이는 대신 *더 나은 대조군을 찾는* 쪽으로 가야 합니다. 실제로 이 사례는 그 때문에 **합성통제법(Synthetic Control)** 의 대표 예제가 되었습니다 — 45개 주를 단순 평균하는 대신, *처치 전 캘리포니아의 추세를 가장 잘 재현하는 가중 평균*을 만들어 대조군으로 쓰는 방법입니다(Abadie, Diamond & Hainmueller, 2010).
>
> ⚠️ **여기에 한계가 하나 더 있습니다 — 처치 단위가 하나뿐입니다(캘리포니아 1개 주).** 통상적인 표준오차·p값은 "처치군 안의 변동"에 기대는데, 처치군이 하나면 그 변동을 추정할 수 없습니다. 그래서 이 사례에서는 **플라시보 검정**(다른 주를 가짜 처치군으로 돌려 효과 분포를 만들고, 실제 추정치가 그 분포에서 얼마나 극단적인지 보기)이 표준적인 추론 방법입니다. 오늘의 과제 '더 나아가기'에서 직접 구현해볼 수 있습니다.
>
> 💡 **정리 — 실데이터에서 평행추세를 다루는 3단계:** ① 그림으로 보여주고 ② 검정으로 뒷받침(또는 반박)하고 ③ 남은 한계를 명시한다. "p < 0.05, 효과 있음"으로 끝내지 않는 것이 준실험 분석의 기본자세입니다.

## ⌨️ 백문이 불여일타! (3) — bad 데이터의 평행추세 그림

```
[문제]
1) 위 bad(평행추세 위반) 데이터로 평행추세 그림을 다시 그려보세요.
2) "이 데이터로 DiD를 쓰면 안 되는 이유"를 한 줄로 적어보세요.
```

In [ ]:
# [C22] ⌨️ 백문이 불여일타! (3) — bad 데이터의 평행추세 그림 그리기

# 여기에 코드를 작성하세요

<details>
<summary>(클릭) 💡 힌트</summary>

- [C17]에서 만든 `trend_bad`는 시점×집단 평균 표입니다 — `trend_bad[0]`이 대조군, `trend_bad[1]`이 처치군의 선입니다.
- `plt.plot()`으로 두 선을 그리고, 처치 시점 표시는 `plt.axvline()`을 쓰면 됩니다 — [C16]·[C17] 그림과 같은 방식입니다.
- 처치 전 구간에서 두 선의 기울기가 나란한지만 보면 판단할 수 있습니다.

</details>

## 🚦 짚고 넘어가기 — Part 6 평행추세

1. 평행추세 가정을 *직접 증명*할 수 없는 이유는 무엇인가요?
2. 그렇다면 무엇으로(어떤 그림·검정으로) 간접 점검하나요?
3. 플라시보 검정에서 효과가 "0이 아니게" 나오면 무엇을 의심해야 하나요?

> 💡 **다음 Part 예고:** DiD 말고도 준실험 도구가 더 있습니다. **RDD·PSM**이 각각 어떤 상황에 맞는지, 코드 없이 개념과 사례로 Part 7에서 살펴봅니다.

# Part 7. RDD·PSM 맛보기 — 다른 준실험 도구들

DiD가 "전후×집단" 구조에 맞는 도구라면, 상황에 따라 더 적합한 준실험 설계가 따로 있습니다. 두 가지를 *개념과 직관 위주*로(코드 없이) 소개합니다 — "이런 상황엔 이런 도구가 있다"를 아는 게 오늘의 목표입니다.

> ❓ **이 파트에서 답할 질문:** 임계값으로 처치가 갈리거나(RDD), 관찰 특성만 다를 때(PSM)는 각각 어떻게 비교 가능한 대조군을 만들까요?

## 회귀불연속 설계 (RDD, Regression Discontinuity)

**언제?** 어떤 *연속 점수*가 *임계값(Cutoff)*을 넘느냐로 처치가 *기계적으로* 갈릴 때.

**직관.** 시험 점수 80점 이상에게 장학금(처치)을 준다고 합시다. **79점 학생과 81점 학생은 사실상 같은 학생**입니다 — 단 2점 차이는 거의 운입니다. 그런데 한 명은 장학금을 받고 한 명은 못 받습니다. 그러니 **임계값 바로 양옆의 학생들을 비교하면, 마치 그 근처에서 무작위 배정이 일어난 것처럼** 처치 효과를 추정할 수 있습니다. "컷오프에서 결과값이 *툭 끊기며(불연속)* 점프하는 크기"가 곧 처치효과입니다.

```text
결과
 ↑                          ●●●●
 |                     ●●●          ← 처치군 (점수 ≥ 컷오프)
 |                    ┊
 |                    ┊  ⇕ 컷오프에서의 불연속 '점프' = 처치효과
 |          ●●●●●     ┊
 |    ●●              ┊
 |    ↑ 비처치군 (점수 < 컷오프)
 +--------------------┊---------------→ 점수(running variable)
                    cutoff
```

**예시.** 장학금 커트라인(점수), 복지 수급 소득 기준선, 선거 당선/낙선 득표율 50% 경계, 약물 처방 기준 수치. **한계.** 컷오프 *근처*의 효과만 알 수 있고(국소적), 사람들이 점수를 조작해 컷오프를 넘나들 수 있으면(Manipulation) 무너집니다.

RDD의 매력은 *직관의 명료함*에 있습니다. "커트라인 바로 위와 아래는 거의 같은 사람들"이라는 한 문장이면 비전공자도 설득되거든요. 실제로 경제학·정책학에서 RDD는 가장 신뢰받는 준실험 설계 중 하나로 꼽힙니다.

### RDD의 필수 점검 — 사람들이 점수를 조작하지 않았나

RDD가 무너지는 대표적인 경우는 **컷오프 근처에서 점수가 조작될 때**입니다. 예컨대 교사가 커트라인 직전 학생의 점수를 살짝 올려주거나, 지원자가 소득을 기준선 아래로 신고하는 경우죠. 그러면 컷오프 양옆이 더 이상 "비슷한 사람들"이 아닙니다.

점검 방법은 의외로 단순합니다 — **점수의 분포(밀도)가 컷오프에서 갑자기 튀는지** 봅니다. 조작이 있으면 한쪽에 관측치가 부자연스럽게 몰립니다. 이것을 **McCrary 밀도 검정**이라 합니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/8/8a/McCrary_%282008%29_Density_Test_on_Data_from_Lee%2C_Moretti%2C_and_Butler_%282004%29.png" width="600" alt="McCrary (2008) Density Test on Data from Lee, Moretti, and Butler (2004).png">

McCrary 밀도 검정의 예(미국 하원 선거 득표율 데이터). 가로축이 컷오프(당선/낙선 경계) 기준 득표 차이, 세로축이 관측치의 밀도입니다. 경계에서 밀도가 **급격히 튀지 않고 매끄럽게 이어지면** 조작이 없다는 근거가 됩니다.

> 🖼️ **그림 출처** — Wikimedia Commons, `McCrary (2008) Density Test on Data from Lee, Moretti, and Butler (2004).png` (CC BY 3.0) · 원본·라이선스 상세: https://commons.wikimedia.org/wiki/File:McCrary_(2008)_Density_Test_on_Data_from_Lee,_Moretti,_and_Butler_(2004).png

> 📚 **실제 RDD 연구 사례** — 미국 하원 선거의 당선/낙선 경계로 현직 효과를 추정한 Lee(2008), 최소 음주 연령 21세 경계로 음주의 사망 영향을 추정한 Carpenter & Dobkin(2009), 학급 규모 상한 규칙(마이모니데스 법칙)을 이용한 Angrist & Lavy(1999)가 대표적입니다. 모두 *"규칙이 만든 임계값"* 을 찾아낸 것이 연구의 핵심이었습니다.

아래에서는 **규칙이 있는 상황을 직접 만들어(시뮬레이션)** RDD를 계산해봅니다. 정답(진짜 효과)을 심어두고 RDD가 그것을 되찾는지 확인하는 방식입니다.

## 예제 · RDD 시뮬레이션 — 컷오프 규칙 심기

In [ ]:
# [C22b] RDD 실습 — 컷오프 규칙을 심은 시뮬레이션으로 원리 확인
# 시나리오: 시험 점수(0~100)가 80점 이상이면 장학금(처치) → 이후 학업 성취도(결과)를 관측
np.random.seed(7)                                                 # 재현성: 난수를 항상 같게 고정
N = 2000
CUTOFF = 80
TRUE_JUMP = 6.0                      # 우리가 심은 진짜 처치효과 (컷오프에서의 점프)

score = np.clip(np.random.normal(75, 12, N), 40, 100)     # running variable(배정 점수)
treated = (score >= CUTOFF).astype(int)                    # 기계적 배정 규칙
# 결과: 점수가 높으면 원래 성취도도 높음(매끄러운 관계) + 처치효과 + 잡음
outcome = 40 + 0.5 * (score - CUTOFF) + TRUE_JUMP * treated + np.random.normal(0, 6, N)
rdd = pd.DataFrame({"score": score, "treated": treated, "outcome": outcome})

# ── (1) 잘못된 방법: 전체를 그냥 두 집단으로 비교 ──
naive_rdd = rdd.loc[rdd.treated == 1, "outcome"].mean() - rdd.loc[rdd.treated == 0, "outcome"].mean()

# ── (2) RDD: 컷오프 근처(bandwidth)만 쓰고, 점수를 통제한 국소 선형회귀 ──
def rdd_estimate(data, bw):
    sub = data[(data.score >= CUTOFF - bw) & (data.score <= CUTOFF + bw)].copy()
    sub["centered"] = sub["score"] - CUTOFF                      # 컷오프를 0으로 이동
    # 양쪽의 기울기가 다를 수 있으니 상호작용까지 넣습니다(국소 선형)
    m = smf.ols("outcome ~ treated + centered + treated:centered", data=sub).fit(cov_type="HC1")  # 이분산에 강한(robust) 표준오차 — 계수는 그대로, SE만 보정
    return m.params["treated"], m.conf_int().loc["treated"].values, len(sub)  # 계수의 95% 신뢰구간

print(f"심은 진짜 효과(컷오프에서의 점프) = {TRUE_JUMP}")
print(f"(1) 단순 두 집단 비교           = {naive_rdd:5.2f}   ← 점수 차이까지 섞여 크게 부풀려짐")
print("\n(2) RDD 국소 선형 추정 — 대역폭(bandwidth)을 바꿔가며")
for bw in [15, 10, 5, 3]:
    est, ci, n_used = rdd_estimate(rdd, bw)
    print(f"    ±{bw:>2}점 구간 (n={n_used:4d}) : {est:5.2f}   95% CI [{ci[0]:5.2f}, {ci[1]:5.2f}]")

## 그림으로 · RDD의 컷오프 점프

In [ ]:
# [C22c] [도식] RDD를 그림으로 — 컷오프에서의 점프
BW = 10
sub = rdd[(rdd.score >= CUTOFF - BW) & (rdd.score <= CUTOFF + BW)].copy()
sub["centered"] = sub["score"] - CUTOFF
m_rdd = smf.ols("outcome ~ treated + centered + treated:centered", data=sub).fit()

# 점수를 구간(bin)으로 묶어 평균을 찍으면 불연속이 눈에 잘 보입니다
rdd["bin"] = (rdd["score"] // 2) * 2 + 1
binned = rdd.groupby("bin").agg(outcome=("outcome", "mean"), n=("outcome", "size")).reset_index()  # 집단별로 묶어서 계산

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))                 # 그림 한 장에 그래프 2개(1행 2열)
for ax, title, xlim in [(axes[0], "All data (binned means)", (40, 100)),
                        (axes[1], f"Zoom: cutoff ± {BW}", (CUTOFF - BW - 1, CUTOFF + BW + 1))]:
    b = binned[(binned["bin"] >= xlim[0]) & (binned["bin"] <= xlim[1])]
    ax.scatter(b["bin"], b["outcome"], s=np.clip(b["n"], 8, 90), color="steelblue", alpha=0.8)
    # 양쪽 적합선
    for side, lo, hi, color in [("left", xlim[0], CUTOFF, "gray"), ("right", CUTOFF, xlim[1], "indianred")]:
        xs = np.linspace(max(lo, CUTOFF - BW) if side == "right" else lo,  # 구간을 균등 간격으로 쪼갠 x축 값
                         hi if side == "right" else min(hi, CUTOFF), 50)
        d = pd.DataFrame({"centered": xs - CUTOFF, "treated": 1 if side == "right" else 0})
        d["treated:centered"] = d["treated"] * d["centered"]
        ax.plot(xs, m_rdd.predict(d), color=color, lw=2)
    ax.axvline(CUTOFF, color="black", ls="--", lw=1)              # 세로 기준선 표시
    ax.set_xlabel("score (running variable)"); ax.set_ylabel("outcome")
    ax.set_title(title); ax.set_xlim(xlim)
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

print(f"컷오프에서의 점프 추정 = {m_rdd.params['treated']:.2f}  (심은 진짜 효과 {TRUE_JUMP})")
print("→ 컷오프 좌우의 적합선이 '툭 끊긴' 그 높이 차이가 처치효과입니다.")

> **읽는 법:** 단순히 "장학금 받은 학생 vs 못 받은 학생"을 비교하면 효과가 크게 부풀려집니다 — 원래 점수가 높은 학생이 처치군에 몰려 있으니까요(선택편향). 반면 **컷오프 근처만 보고 점수를 통제한 RDD 추정치는 심은 진짜 효과(6.0)에 가깝습니다.**
>
> 대역폭(bandwidth)을 바꿔가며 확인한 것도 중요합니다. **좁히면** 컷오프 근처의 "거의 같은 학생들"만 남아 편향이 줄지만 표본이 작아져 신뢰구간이 넓어집니다. **넓히면** 반대입니다 — 정밀하지만 멀리 있는 관측치가 섞여 편향 위험이 커집니다. 이 **편향-분산 트레이드오프**가 RDD 분석의 핵심 판단이고, 실무에서는 여러 대역폭 결과를 함께 보고합니다(강건성).
>
> 오른쪽 그림에서 컷오프 좌우의 두 선이 **끊어진 높이**가 곧 추정치입니다. 이 "툭 끊김"을 보여주는 그림이 RDD 보고의 필수 자료입니다.
>
> ⚠️ **RDD를 쓸 때 점검할 것 세 가지**
> 1. **밀도 점검(McCrary):** 컷오프에서 점수 분포가 튀지 않는가(조작 여부)
> 2. **공변량 연속성:** 컷오프에서 *결과 말고 다른 특성들*(성별·나이 등)도 매끄럽게 이어지는가 — 튀면 다른 요인이 함께 바뀐 것
> 3. **대역폭 강건성:** 대역폭을 바꿔도 결론이 유지되는가
>
> 💡 **실무 도구:** 파이썬에서는 `rdrobust` 패키지(최적 대역폭 자동 선택·편향 보정)가 널리 쓰입니다. 오늘은 원리를 이해하기 위해 `statsmodels`로 직접 계산했습니다.

## 성향점수매칭 (PSM, Propensity Score Matching)

**언제?** 무작위 배정은 없지만, 처치·비처치를 가른 *관찰 가능한 특성들*을 우리가 알고 있을 때.

**직관.** 관찰 데이터에서 처치군과 대조군은 여러 특성(나이·소득·지역 등)이 다릅니다(선택편향). PSM은 각 사람이 *처치받을 확률(성향점수, propensity score)*을 그 특성들로 예측한 뒤, **성향점수가 비슷한 처치자와 비처치자를 짝지어(Matching)** 비교합니다. "특성이 닮은 짝끼리만 비교하니, 처치 여부 외에는 비슷한 사람들"이라는 겁니다 — 무작위 실험에서 두 군이 닮았던 상태를 *사후에 흉내* 내는 셈입니다.

```text
처치군 어떤 사람  ──매칭──▶  성향점수가 가장 비슷한 비처치자
(나이40·소득중·도시)        (나이41·소득중·도시)  ← '거의 같은 사람'끼리 비교
```

**예시.** 특정 마케팅에 노출된 고객 vs 비슷한 미노출 고객, 직업훈련 참가자 vs 비슷한 미참가자. **한계 (결정적!).** PSM은 *측정해서 모델에 넣은* 특성만 맞출 수 있습니다. **측정 못 한 교란(숨은 동기·의지 등)은 여전히 남습니다.** 그래서 PSM은 "측정된 교란만 보정"이라는 강한 한계를 늘 명시해야 합니다 — 앞서 배운 "통제는 측정한 것까지만"과 같은 교훈입니다.

PSM의 절차를 조금 더 풀어보겠습니다. ① 먼저 각 대상이 *처치받을 확률(성향점수)*을 로지스틱 회귀 등으로 추정합니다(여러 특성 → 처치 여부). ② 그 점수가 비슷한 처치자–비처치자를 짝짓거나(매칭), 점수로 가중치를 줍니다. ③ 매칭 후 두 집단의 *특성 분포가 실제로 비슷해졌는지(균형, balance)*를 반드시 점검합니다 — 이게 PSM의 핵심 검증입니다. ④ 균형이 맞으면 비교하고, 안 맞으면 모델을 다시 손봅니다. 여기서도 "결론보다 가정(균형)을 먼저 검증"하는 오늘의 패턴이 반복됩니다.

구체적 예를 들면 — 직업훈련 프로그램의 효과를 보고 싶은데, 참가자는 원래 의욕·학력이 높은 사람들입니다(선택편향). PSM은 미참가자 중에서 *나이·학력·경력·지역이 비슷한* 사람을 참가자마다 짝지어, "닮은 짝끼리"의 취업률을 비교합니다. 마치 무작위 실험에서 두 군이 닮았던 상태를 사후에 재현하는 셈입니다. 다만 거듭 강조하지만, PSM은 *측정해 넣은 변수*에 대해서만 균형을 맞춥니다. "원래의 의욕" 같은 *관측하지 못한* 차이는 그대로 남아, A/B 테스트의 무작위화가 주는 보장과는 근본적으로 다릅니다. 그래서 PSM 결과는 늘 "관측된 특성을 보정했을 때"라는 단서, 그리고 "숨은 교란이 남아 있을 수 있다"는 한계와 함께 보고해야 정직한 분석이 됩니다.

### PSM을 실제 데이터로 — 직업훈련 프로그램의 소득 효과 (`Treatment`)

이제 인과추론 문헌에서 가장 유명한 데이터로 PSM을 직접 해봅니다. 이 사례가 특별한 이유는 **"정답"을 우리가 알고 있다**는 점입니다.

**배경.** 1970년대 미국의 직업훈련 프로그램(NSW)은 참가자를 **무작위로** 배정한 실험이었습니다. 그 실험 결과 훈련의 소득 효과는 **연 약 +1,800달러**로 추정됐습니다 — 이것이 우리의 *벤치마크(정답에 가까운 값)* 입니다.

그런데 LaLonde(1986)는 도발적인 질문을 던졌습니다 — **"실험의 대조군을 버리고, 대신 일반 인구 조사(PSID)에서 뽑은 사람들을 대조군으로 쓰면 어떤 결과가 나올까?"** 즉 *관찰 데이터 상황을 인공적으로 만들어* 준실험 방법이 실험 결과를 복원할 수 있는지 시험한 것입니다.

우리가 쓰는 `Treatment` 데이터가 바로 그 구성입니다 — 훈련 참가자 185명(실험의 처치군) + 일반 인구에서 뽑은 비교집단 2,490명.

| 변수 | 뜻 |
| --- | --- |
| `treat` | 훈련 참가 여부 (처치) |
| `re78` | 1978년(훈련 후) 연간 소득 — **결과** |
| `re74`, `re75` | 훈련 *이전* 연간 소득 — 사전 공변량 |
| `age`, `educ`, `ethn`, `married` | 나이·교육연수·인종·혼인 — 사전 공변량 |

## PSM [1] · 단순 비교는 얼마나 빗나가나

In [ ]:
# [C23b] PSM 1단계 — 단순 비교가 얼마나 빗나가는지 먼저 보기
df_ps = nsw.copy()
df_ps["treat"] = df_ps["treat"].astype(int)
df_ps["married"] = df_ps["married"].astype(int)
df_ps["black"] = (df_ps["ethn"] == "black").astype(int)
df_ps["hispanic"] = (df_ps["ethn"] == "hispanic").astype(int)

BENCHMARK = 1794          # 무작위 실험에서 얻은 효과 추정치(달러) — 우리의 기준점

naive = (df_ps.loc[df_ps.treat == 1, "re78"].mean()
         - df_ps.loc[df_ps.treat == 0, "re78"].mean())
print(f"실험 벤치마크(무작위 배정) ATT ≈ ${BENCHMARK:+,}")
print(f"관찰 데이터 단순 비교      ATT = ${naive:+,.0f}   ← 부호까지 뒤집혔습니다!")

print("\n[왜 이렇게 빗나갔나] 두 집단의 사전 특성 비교")
covs = ["age", "educ", "black", "hispanic", "married", "re74", "re75"]
print(df_ps.groupby("treat")[covs].mean().round(1))               # 집단별로 묶어서 계산

> **읽는 법:** 단순 비교는 **−15,000달러**, 즉 "훈련을 받으면 소득이 크게 *줄어든다*"는 황당한 결론을 냅니다. 표를 보면 이유가 분명합니다 — 훈련 참가자는 비교집단보다 훨씬 젊고, 교육 연수가 짧고, **훈련 이전 소득(re74·re75)이 압도적으로 낮습니다.** 애초에 취업이 어려운 사람들이 프로그램에 참여했으니까요(선택편향의 교과서적 사례).
>
> 즉 두 집단은 "처치 여부 말고도" 근본적으로 다릅니다. PSM은 이 차이를 **비슷한 사람끼리 짝지어** 줄이려는 시도입니다.

## PSM [2] · 성향점수 추정 → 매칭 → 균형 점검

In [ ]:
# [C23c] PSM 2단계 — 성향점수 추정 → 매칭 → 균형 점검 → ATT 재추정
X = df_ps[covs].astype(float)
y = df_ps["treat"]

# (1) 성향점수(처치받을 확률) 추정 — 로지스틱 회귀
ps_model = LogisticRegression(max_iter=5000)                      # PSM 1단계: 성향점수(처치받을 확률) 추정
ps_model.fit(X, y)
df_ps["ps"] = ps_model.predict_proba(X)[:, 1]                     # 각 대상이 처치받을 확률(성향점수)

# (2) 1:1 최근접 이웃 매칭 (캘리퍼 0.05 — 성향점수가 이보다 멀면 짝을 버림)
CALIPER = 0.05
trt = df_ps[df_ps.treat == 1].reset_index(drop=True)
ctl = df_ps[df_ps.treat == 0].reset_index(drop=True)
nn = NearestNeighbors(n_neighbors=1).fit(ctl[["ps"]])             # PSM 2단계: 성향점수가 가장 가까운 짝 찾기
dist, idx = nn.kneighbors(trt[["ps"]])                            # 가장 가까운 짝의 거리와 위치를 반환
keep = dist.ravel() <= CALIPER
matched_trt = trt[keep].reset_index(drop=True)
matched_ctl = ctl.iloc[idx.ravel()[keep]].reset_index(drop=True)
print(f"매칭 결과: 처치군 {len(trt)}명 중 {len(matched_trt)}명이 짝을 찾음 "
      f"(캘리퍼 {CALIPER} 밖은 제외)")

# (3) 균형 점검 — 표준화 평균차(SMD). |SMD| < 0.1 이면 '균형 잡혔다'고 봅니다
def smd(a, b):
    s = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return (a.mean() - b.mean()) / s if s > 0 else 0.0

balance = pd.DataFrame({
    "처치군 평균":     [matched_trt[c].mean() for c in covs],
    "대조군(매칭 전)":  [ctl[c].mean() for c in covs],
    "대조군(매칭 후)":  [matched_ctl[c].mean() for c in covs],
    "|SMD| 매칭 전":   [abs(smd(trt[c], ctl[c])) for c in covs],
    "|SMD| 매칭 후":   [abs(smd(matched_trt[c], matched_ctl[c])) for c in covs],
}, index=covs).round(3)
print("\n[균형 점검 — 매칭이 두 집단을 실제로 닮게 만들었나]")
print(balance.to_string())

# (4) ATT 재추정
att_psm = matched_trt["re78"].mean() - matched_ctl["re78"].mean()
diff = matched_trt["re78"].values - matched_ctl["re78"].values
se_psm = diff.std(ddof=1) / np.sqrt(len(diff))
print(f"\n[결과] 매칭 후 ATT = ${att_psm:+,.0f}  (단순 표준오차 ±${1.96*se_psm:,.0f})")
print(f"       실험 벤치마크 = ${BENCHMARK:+,}")
print(f"       단순 비교     = ${naive:+,.0f}")

## PSM [3] · 성향점수 겹침과 love plot

In [ ]:
# [C23d] [도식] 성향점수 분포의 겹침(overlap)과 매칭 효과
fig, axes = plt.subplots(1, 2, figsize=(13, 4))                   # 그림 한 장에 그래프 2개(1행 2열)

axes[0].hist(ctl["ps"], bins=40, alpha=0.6, color="gray", density=True, label="control (all)")
axes[0].hist(trt["ps"], bins=40, alpha=0.6, color="steelblue", density=True, label="treated")
axes[0].set_xlabel("propensity score"); axes[0].set_ylabel("density")
axes[0].set_title("Before matching: little overlap"); axes[0].legend(fontsize=8)

axes[1].hist(matched_ctl["ps"], bins=30, alpha=0.6, color="gray", density=True, label="control (matched)")
axes[1].hist(matched_trt["ps"], bins=30, alpha=0.6, color="steelblue", density=True, label="treated (matched)")
axes[1].set_xlabel("propensity score"); axes[1].set_title("After matching: distributions overlap")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

fig, ax = plt.subplots(figsize=(8, 3.2))                          # 그래프 하나를 그릴 도화지 준비
yy = np.arange(len(covs))                                         # 정수 눈금 배열 생성
ax.scatter(balance["|SMD| 매칭 전"], yy, color="indianred", label="매칭 전", s=45)
ax.scatter(balance["|SMD| 매칭 후"], yy, color="seagreen", label="매칭 후", s=45)
ax.axvline(0.1, color="gray", ls="--", label="기준선 0.1")           # 세로 기준선 표시
ax.set_yticks(yy); ax.set_yticklabels(covs); ax.set_xlabel("|표준화 평균차(SMD)|")
ax.set_title("Covariate balance: love plot"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력

> **읽는 법:** 결과가 극적입니다.
>
> | 방법 | ATT 추정치 |
> | --- | --- |
> | 관찰 데이터 단순 비교 | 약 **−15,200달러** (부호가 반대!) |
> | **PSM(성향점수 매칭)** | 약 **+1,450달러** |
> | 무작위 실험 벤치마크 | 약 **+1,800달러** |
>
> 매칭만으로 **−15,200에서 +1,450으로**, 실험 결과와 같은 방향·비슷한 크기까지 왔습니다. 왼쪽 그림이 그 이유를 보여줍니다 — 매칭 전에는 두 집단의 성향점수 분포가 거의 겹치지 않지만(다른 사람들!), 매칭 후에는 잘 겹칩니다.
>
> ⚠️ **그런데 love plot을 정직하게 읽어야 합니다.** `educ`·`black`·`married`·`re75`는 |SMD|가 0.1 아래로 잘 내려왔지만, **`age`와 `re74`는 여전히 0.2를 넘습니다.** 즉 **균형이 완전히 잡히지 않았습니다.** 이런 상태에서 "매칭했으니 비교 가능하다"고 단정하면 안 되고, ① 공변량을 추가·변형하거나(제곱항·로그) ② 캘리퍼를 좁히거나 ③ 다른 매칭 방식(마할라노비스 거리, 가중치 방식인 IPW)을 시도해봐야 합니다. 아래 셀에서 몇 가지 설정을 바꿔 결론이 얼마나 흔들리는지 확인합니다.
>
> 또 하나 — 매칭 후 ATT의 표준오차가 커서 **신뢰구간이 0을 포함**합니다. "실험 벤치마크와 비슷한 값을 얻었다"는 것과 "그 값이 0과 유의하게 다르다"는 것은 별개의 이야기입니다. 185명이라는 작은 처치군에서는 정밀도 자체가 낮습니다.

## PSM [4] · 설정을 바꿔도 결론이 유지되나

In [ ]:
# [C23e] PSM 민감도 점검 — 설정을 바꿔도 결론이 유지되나
def psm_att(cov_list, caliper):
    """공변량 목록과 캘리퍼를 받아 (매칭 수, ATT, 최대 |SMD|)를 돌려준다."""
    Xs = df_ps[cov_list].astype(float)
    ps = LogisticRegression(max_iter=8000).fit(Xs, df_ps["treat"]).predict_proba(Xs)[:, 1]
    dd = df_ps.assign(ps_tmp=ps)
    t_, c_ = dd[dd.treat == 1].reset_index(drop=True), dd[dd.treat == 0].reset_index(drop=True)
    nn_ = NearestNeighbors(n_neighbors=1).fit(c_[["ps_tmp"]])
    dist_, idx_ = nn_.kneighbors(t_[["ps_tmp"]])
    k_ = dist_.ravel() <= caliper
    mt_, mc_ = t_[k_].reset_index(drop=True), c_.iloc[idx_.ravel()[k_]].reset_index(drop=True)
    max_smd = max(abs(smd(mt_[v], mc_[v])) for v in cov_list)
    return len(mt_), mt_["re78"].mean() - mc_["re78"].mean(), max_smd

covs_plus = covs + ["u74", "u75"]        # 훈련 이전 '실업 여부' 더미를 추가
df_ps["u74"] = df_ps["u74"].astype(int)
df_ps["u75"] = df_ps["u75"].astype(int)

print(f"{'설정':34s} {'매칭 수':>7s} {'ATT($)':>10s} {'최대|SMD|':>10s}")
print("-" * 66)
for cl, cal, label in [(covs, 0.05, "기본 공변량, 캘리퍼 0.05"),
                       (covs, 0.02, "기본 공변량, 캘리퍼 0.02"),
                       (covs_plus, 0.02, "+ 실업 더미(u74,u75), 캘리퍼 0.02"),
                       (covs_plus, 0.01, "+ 실업 더미(u74,u75), 캘리퍼 0.01")]:
    n_, att_, mx_ = psm_att(cl, cal)
    print(f"{label:34s} {n_:>7d} {att_:>10,.0f} {mx_:>10.3f}")
print(f"\n실험 벤치마크 = ${BENCHMARK:,}   /   단순 비교 = ${naive:,.0f}")
print("→ 설정을 바꿔도 ATT가 +1,200~+1,700 범위에 머물면, 결론이 특정 설정에 좌우되지 않는다는 근거가 됩니다.")

> **읽는 법:** 네 가지 설정에서 ATT가 대략 **+1,200 ~ +1,700달러** 범위에 머무릅니다. 실험 벤치마크(+1,800)와 같은 방향·비슷한 크기이고, 무엇보다 **단순 비교의 −15,200과는 비교할 수 없이 가깝습니다.** 설정을 바꿔도 결론이 흔들리지 않는 것 — 이것이 *강건성(robustness)* 의 증거입니다.
>
> 균형(최대 |SMD|)은 설정에 따라 좋아지거나 나빠집니다. 캘리퍼를 좁히면 균형은 개선되지만 짝을 찾지 못하는 처치군이 늘어(매칭 수 감소) 결론이 *어떤 사람들에 대한 것인지*가 달라집니다. **공짜 점심은 없습니다** — 그래서 PSM 보고서에는 *선택한 설정과 그 근거, 그리고 다른 설정에서의 결과*를 함께 적습니다.
>
> 💡 **더 나은 대안들:** 매칭 대신 성향점수를 *가중치*로 쓰는 **IPW(역확률가중)**, 회귀와 가중치를 결합해 둘 중 하나만 맞아도 되는 **이중강건(Doubly Robust) 추정**, 매칭 후 회귀로 남은 불균형을 보정하는 방법 등이 널리 쓰입니다. 오늘은 원리를 손으로 확인하는 데 집중했습니다.

## 세 준실험 도구 — 언제 무엇을

| 도구 | 적합한 상황 | 핵심 가정 | 한계 |
|------|-------------|-----------|------|
| **DiD** | 처치 전후 + 처치/대조군 데이터 | 평행추세 | 추세가 갈리면 무효 |
| **RDD** | 임계값으로 처치가 갈림 | 컷오프 근처 연속성 | 국소 효과, 점수 조작에 취약 |
| **PSM** | 관찰 특성으로 매칭 가능 | 측정된 교란이 전부 | 숨은 교란 보정 불가 |

> 💡 공통점이 보이시나요? 셋 다 **"비교 가능한 대조군을 어떻게 구하느냐"**의 서로 다른 답입니다. DiD는 *처치 전 추세*를, RDD는 *컷오프 반대편*을, PSM은 *닮은 특성의 짝*을 대조군으로 씁니다. 그리고 셋 다 각자의 *가정*이 깨지면 무너집니다.

## ⌨️ 백문이 불여일타! (4) — 세 상황에 맞는 준실험 도구 고르기

```
[문제]
다음 세 상황에 가장 적합한 준실험 도구(DiD / RDD / PSM)를 골라보세요. (코드 없이, 이유와 함께)
1) 소득 기준선 이하 가구에만 지급된 보조금의 효과
2) 한 지역에만 새 정책이 시행됐고, 인근 지역은 그대로일 때 전후 데이터가 있음
3) 어떤 캠페인에 스스로 반응한 고객과 비슷한 특성의 미반응 고객을 비교하고 싶을 때
```

In [ ]:
# [C23f] ⌨️ 백문이 불여일타! (4) — 세 상황에 맞는 준실험 도구 고르기

# 코드는 필요 없습니다 — 여기에 각 상황의 선택(DiD/RDD/PSM)과 근거를 주석으로 작성해보세요
# 1)
# 2)
# 3)

<details>
<summary>(클릭) 💡 힌트</summary>

- 처치가 *무엇으로* 갈렸는지부터 보세요 — 임계값인가요, 특정 지역·시점인가요, 스스로 선택했나요?
- 바로 위 "세 준실험 도구 — 언제 무엇을" 표의 각 행과 상황을 짝지어보세요.
- 각 도구가 서 있는 가정(컷오프 근처 연속성·평행추세·측정된 교란이 전부)이 그 상황에서 그럴듯한지도 함께 점검해보세요.

</details>

## 🚦 짚고 넘어가기 — Part 7 RDD·PSM

1. RDD는 어떤 구조(무엇으로 처치가 갈릴 때)에 적합한가요?
2. PSM의 *결정적 한계*는 무엇인가요?
3. DiD·RDD·PSM의 공통된 목표를 한 문장으로 말해보세요.

> 💡 **다음 Part 예고:** 도구가 여럿이지만, 더 중요한 건 *태도*입니다. 모든 준실험을 관통하는 한 가지 원칙 — "가정 먼저 검증"을 Part 8에서 정리합니다.

# Part 8. 가정을 먼저 검증한다 — 인과 분석가의 태도

오늘 배운 DiD·RDD·PSM은 서로 다른 기법이지만, 하나의 태도로 묶입니다. **"결론을 말하기 전에, 그 결론이 기댄 가정을 먼저 검증하라."** 이것이 인과추론 전체를 관통하는 정신이고, 데이터로 의사결정하는 사람의 직업윤리이기도 합니다.

> ❓ **이 파트에서 답할 질문:** 인과 분석에서 "가정을 먼저 검증한다"는 태도는 구체적으로 무엇을 하라는 뜻일까요?

## 모든 인과 도구는 '가정' 위에 서 있다

| 도구 | 그 위에 선 핵심 가정 | 가정이 깨지면 |
|------|----------------------|---------------|
| 무작위 실험(A/B) | 무작위 배정·SRM 정상 | 교란이 다시 섞임 |
| DiD | 평행추세 | 가짜 효과 |
| RDD | 컷오프 근처 연속성 | 점프를 효과로 오해 |
| PSM | 측정된 교란이 전부 | 숨은 교란 남음 |
| 다중 회귀 | 빠진 교란 없음 | 누락변수 편향 |

어떤 도구도 *가정 없이는* 인과를 말하지 못합니다. 그래서 인과 분석의 품질은 *어떤 기법을 썼느냐*보다 **가정을 얼마나 정직하게 점검하고 밝혔느냐**로 갈립니다.

## "가정 먼저 검증"의 실천

1. **가정을 명시한다.** "이 결론은 평행추세가 성립한다는 가정에 기댄다"를 보고서에 *대놓고* 적습니다.
2. **점검 가능한 증거를 댄다.** 처치 전 추세 그림, 플라시보 검정, 매칭 균형표 등 *간접 증거*를 보여줍니다.
3. **민감도를 따진다.** "가정이 조금 깨지면 결론이 얼마나 흔들리나?"를 점검합니다.
4. **가정이 깨지면 멈추거나 보강한다.** 평행추세가 명백히 위반이면, DiD 결과를 *밀어붙이지 않습니다.* 다른 설계를 찾거나 "인과는 말할 수 없다"고 정직하게 보고하는 것도 훌륭한 분석입니다.

> 💡 초보와 전문가의 차이가 여기서 갈립니다. 초보는 "효과를 찾았다!"에 흥분하지만, 전문가는 "이 효과가 *진짜이려면* 무엇이 참이어야 하지?"를 먼저 묻고, 그 가정을 *스스로 공격*해봅니다. 살아남은 결론만 신뢰합니다. 통계는 *확신을 파는 도구*가 아니라 *불확실성을 정직하게 다루는 도구*입니다.

구체적인 예를 하나 들어보겠습니다. 어떤 분석가가 "신규 매장 디자인이 매출을 12% 올렸다(DiD)"고 보고합니다. 미숙한 보고는 여기서 끝납니다. 반면 성숙한 보고는 이렇게 이어집니다 — "단, 이 추정은 *평행추세 가정*에 기댄다. 처치 전 6개월간 두 집단 매출이 나란히 움직였음을 그림으로 확인했고(첨부), 가짜 처치 시점 플라시보 검정도 통과했다. 다만 같은 시기 인근에 대형 경쟁점이 폐업한 점은 잠재적 교란이며, 이를 통제하지 못했으므로 효과가 다소 과대평가됐을 수 있다." 어느 쪽이 더 신뢰가 가나요? *한계를 아는 분석가*입니다. 가정과 한계를 밝히는 건 분석을 약하게 만드는 게 아니라, *정확히 어디까지 믿을 수 있는지*를 알려주는 — 데이터 전문가의 가장 큰 미덕입니다. 이 태도는 오늘 모듈에서 배운 모든 도구를 관통합니다. 도구가 화려할수록, 그 도구가 선 가정을 더 겸손하게 따져야 합니다.

## 🚦 짚고 넘어가기 — Part 8 가정 먼저 검증

1. DiD·RDD·PSM·회귀는 각각 어떤 가정 위에 서 있나요?
2. "가정 먼저 검증"을 실천하는 네 단계를 말해보세요.
3. 가정이 명백히 깨졌을 때, 분석가가 취할 수 있는 정직한 선택은?

> 💡 **다음 Part 예고:** 배운 걸 모아, 합성 정책 데이터로 DiD를 *추정→평행추세 점검→보고*까지 수행하고, 모듈 전체를 돌아보는 종합 실습으로 마무리합니다.

# 🧪 종합 실습 — 정책 효과를 DiD로 (추정·검증·보고)

한 시(市)가 특정 지역에만 상권 활성화 정책을 시행했습니다(처치). 인근 지역은 그대로고요(대조). *진짜 효과(ATT)를 심은* 합성 데이터로, **추정 → 평행추세 점검 → 의사결정 보고서**의 전 과정을 수행합니다.

```text
[1] 평행추세 점검 — DiD를 쓸 자격이 있나? (가정 먼저!)
[2] DiD 회귀 — ATT 추정·신뢰구간
[3] 플라시보 검정 — 결론의 견고함
[4] 의사결정 보고서 — "정책에 효과가 있었나" (가정 검증 포함)
```

## 종합 [0] · 합성 정책 데이터 만들기

In [ ]:
# [C24] 합성 정책 데이터 (평행추세 성립 + ATT 심음)
np.random.seed(2024)                                              # 재현성: 난수를 항상 같게 고정
times = np.array([-3, -2, -1, 0, 1, 2])
def make(slope, base, att, treat):
    rows = []
    for t in times:
        eff = att if (treat and t >= 0) else 0
        rows.append(pd.DataFrame({"time": t, "treat": int(treat),
                                  "y": base + slope*t + eff + np.random.normal(0, 5, 250)}))
    return pd.concat(rows)
TRUE = 16
policy = pd.concat([make(4, 120, 0,    False),   # 대조 지역
                    make(4, 105, TRUE, True)])   # 처치 지역
tr = policy.groupby(["time", "treat"])["y"].mean().unstack()      # 집단별로 묶어서 계산
print("시점별 평균:"); print(tr.round(1))

## 종합 [1] · 평행추세 점검 (가정 먼저!)

In [ ]:
# [C25] [1] 평행추세 점검 (가정 먼저!)
fig, ax = plt.subplots(figsize=(9, 4))                            # 그래프 하나를 그릴 도화지 준비
ax.plot(tr.index, tr[0], "o-", color="gray", label="control")
ax.plot(tr.index, tr[1], "o-", color="steelblue", label="treated")
ax.axvline(-0.5, color="indianred", ls="--", label="policy starts")  # 세로 기준선 표시
ax.set_title("[1] Parallel pre-trends check"); ax.set_xlabel("time"); ax.set_ylabel("mean y"); ax.legend()
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력
print("처치 전(-3~-1) 두 선이 나란하면 → DiD 적용 자격 OK")

## 종합 [2][3] · DiD 회귀와 플라시보

In [ ]:
# [C26] [2] DiD 회귀 + [3] 플라시보
policy["post"] = (policy["time"] >= 0).astype(int)
m = smf.ols("y ~ treat + post + treat:post", data=policy).fit()   # DiD 회귀 — treat:post 상호작용 계수가 곧 ATT
att = m.params["treat:post"]; ci = m.conf_int().loc["treat:post"]  # 계수의 95% 신뢰구간
print(f"[2] DiD ATT = {att:.2f}, 95% CI [{ci[0]:.2f}, {ci[1]:.2f}], p = {m.pvalues['treat:post']:.4f}  (심은 값 {TRUE})")

pre = policy[policy["time"] < 0].copy()
pre["fpost"] = (pre["time"] >= -1).astype(int)
mp = smf.ols("y ~ treat + fpost + treat:fpost", data=pre).fit()
print(f"[3] 플라시보(가짜 처치) ATT = {mp.params['treat:fpost']:.2f}, p = {mp.pvalues['treat:fpost']:.3f}  (0 근처·비유의면 OK)")

## 📋 정책 효과 DiD 보고서 (가정 검증 포함)

위 결과로 아래 보고서를 채웁니다. **1번(가정)을 2번(결과)보다 먼저** 쓰는 게 인과 보고서의 핵심입니다.

```markdown
# 상권 활성화 정책 효과 DiD 분석 보고서

## 1. 식별 가정 검증  ← 결론보다 먼저!
- 평행추세: 처치 전 3개 시점에서 두 지역이 나란히 변함 (그림 첨부) → 가정 그럴듯
- 플라시보 검정: 가짜 처치 시점 효과 = __ (p = __) → 0 근처/비유의 → 통과

## 2. 효과 추정 (DiD)
- ATT = __, 95% CI [__, __], p = __
- 해석: "정책 시행 후 처치 지역 매출이 대조 지역 대비 평균 __ 더 증가"

## 3. 의사결정 제안
- (예) "효과가 유의하고 평행추세·플라시보를 통과하므로, 정책의 긍정적 효과를 지지.
   타 지역 확대 검토 가능" 또는 "가정 점검 미흡 시 추가 데이터 권고"

## 4. 한계
- 관찰 데이터 준실험 → 평행추세 가정에 의존. 동시기 다른 정책·지역 특수성 가능.
- 효과의 지속성은 더 긴 사후 기간 관찰 필요.
```

> 💡 빈칸(`__`)은 위 출력으로 채우세요. 특히 **1번 가정 검증을 통과하지 못하면, 2번 효과는 의미가 없다**는 점 — 오늘 모듈 전체의 결론입니다.

## 종합 — 이벤트 스터디로 정책 효과의 '동태'까지

종합 데이터에도 이벤트 스터디를 적용해, 효과가 *시점에 따라 어떻게* 나타나는지 봅니다. 정책은 보통 즉시·일정하게 효과를 내기보다, 서서히 커지거나 한 번에 점프하기도 합니다. 그리고 결론이 *특정 시점 선택*에 흔들리지 않는지 강건성도 점검합니다.

## 종합 · 이벤트 스터디로 동적 효과 보기

In [ ]:
# [C27] 종합 이벤트 스터디 — 시점별 동적 효과
es2 = policy.groupby(["time", "treat"])["y"].mean().unstack()     # 집단별로 묶어서 계산
es2["rel"] = (es2[1] - es2[0]) - (es2[1] - es2[0]).loc[-1]
fig, ax = plt.subplots(figsize=(9, 4))                            # 그래프 하나를 그릴 도화지 준비
ax.plot(es2.index, es2["rel"], "o-", color="purple")
ax.axhline(0, color="gray"); ax.axvline(-0.5, color="indianred", ls="--", label="policy")  # 세로 기준선 표시
ax.set_title("Policy event study: dynamic effect"); ax.set_xlabel("time"); ax.set_ylabel("rel. gap"); ax.legend()
plt.tight_layout(); plt.show()                                    # 그래프 간격 자동 정리 후 출력
print("처치 전 ~0(평행), 처치 후 양수로 점프 → 정책 효과의 시점별 크기")

> **읽는 법(그림):** 정책 전 평평(평행추세 OK), 정책 후 효과가 나타납니다. 시점별 높이로 "효과가 즉시 나왔나, 점점 커졌나"까지 읽을 수 있습니다 — 단일 ATT 숫자보다 풍부한 이야기입니다.

## 강건성 · 사후 시점을 달리 잡아도 안정적인가

In [ ]:
# [C28] 강건성 — 사후 시점을 달리 잡아도 ATT가 안정적인가 (간단 민감도)
for cut in [0, 1, 2]:
    sub = policy[(policy["time"] < 0) | (policy["time"] == cut)].copy()
    sub["post"] = (sub["time"] == cut).astype(int)
    a = smf.ols("y ~ treat + post + treat:post", data=sub).fit().params["treat:post"]  # DiD 회귀 — treat:post 상호작용 계수가 곧 ATT
    print(f"사후 시점 t={cut}만 사용 시 DiD ATT = {a:.1f}")
print(f"(심은 진짜 ATT = {TRUE})  → 어느 사후 시점을 써도 ATT가 안정적이면 결론이 견고합니다.")

> **읽는 법:** 사후 기간을 달리 잡아도 ATT 추정이 진짜 값 근처에서 안정적이면, 결론이 특정 시점 선택에 좌우되지 않는다는 *강건성*의 증거입니다. 인과 분석가는 이렇게 *결론을 여러 각도에서 흔들어보고* 살아남는지 확인합니다.

## 강건성 · 여러 가짜 시점 플라시보 검정

In [ ]:
# [C29] 추가 강건성 — 처치 전 여러 '가짜 시점'에 플라시보 DiD (모두 0 근처여야 안심)
pre_p = policy[policy["time"] < 0].copy()
print("플라시보 DiD (처치 전 가짜 시점들):")
for fake in [-2, -1]:
    pre_p["fp"] = (pre_p["time"] >= fake).astype(int)
    a = smf.ols("y ~ treat + fp + treat:fp", data=pre_p).fit().params["treat:fp"]
    print(f"  가짜 처치 t={fake}: ATT = {a:+.2f}  (0 근처면 평행추세 지지)")
print("→ 여러 가짜 시점에서 모두 효과가 0 근처면, 진짜 처치 후 효과를 더 믿을 수 있습니다.")

> **읽는 법:** 처치 *전*의 여러 가짜 시점에 플라시보 DiD를 돌려 모두 0 근처가 나오면, "우리 DiD가 잡은 진짜 효과는 평행추세 위반의 산물이 아니다"라는 확신이 커집니다. 결론을 *여러 번, 여러 방식으로* 흔들어보는 것 — 그게 신뢰받는 인과 분석의 자세입니다.

## ⌨️ 백문이 불여일타! (5) — 처치군 기울기 8로 평행추세 위반 실험

```
[문제]
1) 위 종합 데이터에서 처치군의 시간 기울기를 4 → 8로 바꿔(평행추세 위반) DiD를 다시 돌려보세요.
2) ATT 추정이 진짜 값(16)에서 얼마나 빗나가는지 확인해보세요.
3) 평행추세 그림에서 그 위반이 보이는지도 확인해보세요.
```

In [ ]:
# [C30] ⌨️ 백문이 불여일타! (5) — 처치군 기울기 8로 평행추세 위반 실험

# 여기에 코드를 작성하세요

<details>
<summary>(클릭) 💡 힌트</summary>

- [C24]에서 정의한 `make()`를 그대로 쓰되, 처치군 쪽 호출의 기울기 인자만 4에서 8로 바꿔 데이터를 다시 만들어보세요.
- `post` 열을 만들고 `smf.ols("y ~ treat + post + treat:post", ...)`로 DiD를 돌리면 됩니다 — [C26]과 같은 흐름입니다.
- 시점×집단 평균 표를 `groupby(["time", "treat"])`로 만들어 그리면, 처치 전 기울기 차이가 눈에 보입니다.

</details>

## 결과를 제대로 보고하는 법 — 표준 표기와 흔한 실수

준실험 인과 결과는 *가정 검증과 함께* 보고합니다.

| 항목 | 표준 보고 형식 (예) |
|------|---------------------|
| DiD 효과 | "정책 후 처치군이 대조군 대비 `+16.12`(95% CI [15.27, 16.98], p<.001) 증가" |
| 가정 검증 | "처치 전 3시점 평행추세 확인 + 플라시보 효과 비유의(p = .95)" |
| 한계 | "관찰 데이터·평행추세 의존, 동시기 타 정책 가능성" |

*위 숫자는 이 노트북 종합 실습 [C26]에서 실제로 계산한 값입니다. 보고서에 쓸 숫자는 반드시 자기 분석 출력에서 가져오세요.*

> 📌 **흔한 실수 5가지 (제출 전 체크):**
> 1. **평행추세 점검 없이 DiD 결과만 보고** → 가짜 효과일 수 있음(반드시 처치 전 추세 그림)
> 2. **단순 전후·집단 비교를 DiD라 착각** → "차이의 차이" 구조인지 확인
> 3. **표준오차를 잘못 잡음** → 패널이면 클러스터 SE 고려
> 4. **RDD·PSM의 한계를 안 밝힘** → RDD는 국소 효과, PSM은 측정된 교란만 보정
> 5. **결론을 단정적 인과로** → "이 가정이 성립한다면"이라는 조건을 늘 붙이기

> 💡 **한 문장 원칙:** *가정을 먼저 검증해 보이고(평행추세·플라시보), 효과는 신뢰구간과 함께, 결론은 "이 가정 하에서"라는 조건과 함께 보고하라.*

# ❓ 오늘의 퀴즈

배운 내용을 잠깐 확인해보겠습니다. 틀려도 괜찮습니다.

## 개념 퀴즈

1. 전후 비교와 집단 비교가 각각 통제하지 못하는 교란은 무엇인가요?
2. DiD 회귀 `y ~ treat + post + treat:post` 에서 ATT는 어느 계수인가요?
3. DiD의 핵심 식별 가정의 이름은 무엇이고, 어떻게 *간접* 점검하나요?
4. 임계값으로 처치가 갈릴 때 적합한 준실험 도구는?

## 코드 퀴즈

아래 4칸 평균이 주어졌을 때 DiD를 손으로 계산하세요. (대조군 전 50 → 후 58, 처치군 전 70 → 후 90)

In [ ]:
# [C31] 코드 퀴즈 — 네 칸 평균으로 DiD 손 계산

# 여기에 코드를 작성하세요

# 🎓 오늘의 정리

## 오늘 배운 것의 연결

```text
[Part 3] 단순 비교의 함정 (전후=시간 / 집단=기존차이)
   ↓ 둘을 결합해
[Part 4] DiD = 차이의 차이 (두 교란 동시 제거)
   ↓ 회귀로
[Part 5] treat:post 계수 = ATT (CI·p·공변량)
   ↓ 단, 생명은
[Part 6] 평행추세 (처치 전 추세·플라시보로 점검)
   ↓ 다른 도구들
[Part 7] RDD(임계값) · PSM(매칭)
   ↓ 관통하는 태도
[Part 8] 가정 먼저 검증
```

## 한 장 정리

| 개념 | 한 줄 요약 |
|------|-----------|
| 준실험 | 실험 없이 '비교 가능한 대조군'을 영리하게 구함 |
| DiD | (처치군 후−전) − (대조군 후−전) = ATT |
| DiD 회귀 | `treat:post` 상호작용 계수 = ATT |
| 평행추세 | DiD의 생명. 처치 전 추세·플라시보로 점검 |
| RDD | 임계값 양옆 비교(국소 효과) |
| PSM | 닮은 특성끼리 매칭(측정된 교란만) |
| 인과 분석가의 태도 | 결론보다 가정을 먼저 검증 |

> 💡 **다음 모듈 예고:** 통계·인과추론 모듈이 여기서 끝납니다. 다음은 **머신러닝** — 오늘까지 배운 '관계'(회귀)를 *예측*으로 본격 확장합니다. 지도학습, 선형·로지스틱 회귀, 모델 평가로 나아가요. 인과(왜)와 예측(무엇이 일어날까)은 다른 질문이지만, 둘 다 오늘까지 쌓은 통계 위에 섭니다.

## 🗺️ 모듈 전체 회고 — 앞서부터 오늘까지

긴 여정이었습니다. 통계·인과추론 모듈에서 우리가 쌓아온 것을 한눈에 돌아보겠습니다.

| 주제 | 핵심 질문 | 얻은 도구 |
|------|-----------|-----------|
| **기술통계·분포** | 데이터는 어떻게 생겼나? | 기술통계·분포·왜도/첨도 |
| **t-검정·ANOVA** | 두(여러) 집단 평균이 다른가? | t-test·ANOVA·효과크기·검정력 |
| **카이제곱·비모수** | 가정이 깨지면? 범주형은? | 카이제곱·비모수·베이즈 정리 |
| **상관·회귀** | 변수들의 관계는? | 상관·회귀·진단(VIF·잔차) |
| **A/B·인과 입문** | 우리가 만든 변화인가? | A/B·잠재결과·빈도주의/베이지안 |
| **준실험(DiD·RDD·PSM)** | 실험 없이 인과를? | DiD·RDD·PSM·평행추세 |

이 흐름을 관통하는 **하나의 정신**이 있었습니다 — *"데이터로 성급히 단정하지 말고, 적절한 도구를 골라, 가정을 정직하게 따지며 판단하라."* 기술통계의 "분포를 먼저 보라"부터, 검정의 "유의성과 효과크기를 함께 보라", 회귀의 "상관은 인과가 아니다", 인과추론의 "가정을 먼저 검증하라"까지 — 모두 **겸손하고 정직한 데이터 태도**의 다른 얼굴이었습니다.

여러분은 이제 데이터를 *읽고*, *비교하고*, *관계를 모델링하고*, *인과를 신중히 추정하는* 통계적 사고의 한 바퀴를 완주했습니다. 정말 수고 많으셨습니다. 이 사고의 틀은 앞으로 어떤 분석·머신러닝을 배우든 든든한 토대가 될 겁니다. 🌱

마지막으로 한 가지만 더 — 오늘 배운 DiD·RDD·PSM은 *입문*입니다. 각각은 그 자체로 깊은 분야이고, 실무에선 더 정교한 변형(staggered DiD, 합성통제법, 이중강건 추정 등)이 쓰입니다. 하지만 그 모든 정교한 도구도 결국 오늘 배운 *두 가지 질문*에서 출발합니다 — "비교 가능한 대조군을 어떻게 구하나"와 "그 비교가 기댄 가정을 어떻게 검증하나". 이 두 질문을 품고 있다면, 앞으로 어떤 새로운 인과 방법을 만나도 그 핵심을 빠르게 꿰뚫을 수 있을 겁니다. 여러분은 이미 그 출발선에 섰습니다.

# 📝 오늘의 과제

> 짧은 점검은 노트북에서 끝냈습니다. 이제 **관찰 데이터로 준실험을 적용**하는 과제입니다.

## 과제 — 준실험 인과 분석 보고서

### 1. 데이터 — 라이브러리가 제공하는 준실험 데이터에서 고르기

*전후 × 처치/대조* 구조(DiD)나 *매칭 가능한 공변량*(PSM)이 있는 데이터를 고르세요. 아래는 모두 실제 논문에 쓰인 자료입니다.

```python
import statsmodels.api as sm

# ── DiD용 (정책 변화 + 전후 + 두 집단) ──
sm.datasets.get_rdataset("injury", "wooldridge").data     # 켄터키/미시간 산재보험 개정 (본문 예제)
sm.datasets.get_rdataset("kielmc", "wooldridge").data     # 소각장 건설 전후 주택 가격 (1978 vs 1981, nearinc)
sm.datasets.get_rdataset("jtrain", "wooldridge").data     # 훈련 보조금(grant) × 연도 → 불량률
sm.datasets.get_rdataset("Cigar", "plm").data             # 주 × 연도 담배 판매 패널 (본문 이벤트 스터디)
sm.datasets.get_rdataset("Fatalities", "AER").data        # 주 × 연도 음주운전 정책·사망률 패널

# ── PSM용 (처치 + 사전 공변량) ──
sm.datasets.get_rdataset("Treatment", "Ecdat").data       # 직업훈련(NSW/PSID) — 본문 예제
sm.datasets.get_rdataset("Schooling", "Ecdat").data       # 대학 근접성 × 교육연수 → 임금
```

> 💡 **추천 조합:** `kielmc`는 "소각장이 세워질 지역의 주택 가격이 정말 떨어졌나"를 1978·1981 두 시점 × 인접/비인접으로 보는 **가장 깔끔한 2×2 DiD 실습**입니다(`nearinc`, `year`, `rprice`). 본문 예제와 다른 데이터로 연습하고 싶다면 여기서 시작하세요.

### 2. 분석

- [ ] **가정 검증을 먼저.** 여러 시점이 있으면 **처치 전 추세 그림 + 이벤트 스터디**, 두 시점뿐이면 *왜 평행추세가 그럴듯한지* 근거를 글로 제시하세요.
- [ ] **플라시보 검정**(처치 전 가짜 시점) 또는 **가짜 처치군** 검정을 하나 이상 수행하세요.
- [ ] 가정이 그럴듯하면 **DiD 회귀로 ATT**를 추정하고 **신뢰구간**과 함께 보고하세요(패널이면 클러스터 표준오차).
- [ ] (PSM을 골랐다면) **성향점수 추정 → 매칭 → 균형 점검(SMD·love plot) → ATT**의 4단계를 모두 보이세요.
- [ ] (선택) 공변량을 추가해 추정이 얼마나 흔들리는지 **민감도**를 확인하세요.

### 3. 보고서

위 종합 실습 양식으로 정리합니다. **반드시 "1. 식별 가정 검증"을 결과보다 먼저** 쓰고, 가정이 약하면 그 한계를 정직하게 명시하세요.

### 평가 기준

| 축 | 무엇을 보나 |
|----|-------------|
| 정확성 | DiD·PSM 추정과 진단이 올바른가 |
| 합리성 | 가정 검증을 결론보다 우선했는가, 도구 선택이 타당한가 |
| 해석력 | 결과를 "이 가정 하에서"라는 조건과 함께 서술했는가 |
| 인사이트 | 효과를 비즈니스/정책 언어로 신중히(인과 과장 없이) 번역했는가 |

> 🚀 **더 나아가기:** ① 같은 데이터의 처치군을 일부러 비틀어 평행추세를 깨고, DiD 추정이 어떻게 망가지는지 관찰해보세요. ② 캘리포니아 사례처럼 처치 단위가 하나뿐인 데이터에서 **가짜 처치군 플라시보**(다른 주를 처치군으로 돌려 효과 분포를 만들기)를 구현해, 실제 추정치가 그 분포에서 얼마나 극단적인지 확인해보세요 — 합성통제법의 추론 아이디어를 손으로 체험할 수 있습니다.

---

오늘도 한 걸음, 수고하셨습니다! 🎉
어제의 나보다 데이터를 다루는 손이 한 뼘 더 능숙해졌습니다. 다음 시간에 또 만나요.

오늘 여러분은 *실험조차 할 수 없을 때* 인과에 다가가는 법 — 이중차분과 준실험을 배웠고, 무엇보다 **"결론보다 가정을 먼저 검증한다"**는 인과 분석가의 태도를 익혔습니다. 그리고 이번 시간으로 통계·인과추론 모듈 전체를 완주했습니다. 데이터를 읽고, 비교하고, 모델링하고, 인과를 신중히 추정하는 — 통계적 사고의 한 바퀴를 끝까지 돌았습니다. 다음 머신러닝 여정에서, 오늘까지 쌓은 이 단단한 토대 위에 새로운 것을 얹어 갑시다. 통계적 사고라는 든든한 무기를 손에 쥔 여러분이라면 그 다음 여정도 분명 잘 해낼 것입니다. 그동안 정말 고생 많으셨고, 끝까지 완주한 여러분이 자랑스럽습니다. 🌱🚀

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
기획·제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 자료는 생성형 AI를 활용해 제작되었고, 제작자의 검수를 거쳐 완성되었습니다.<br>
무단 복제 및 배포를 금합니다.</sub>